In [2]:
import pandas as pd
import numpy as np

# ── TRAIN ──────────────────────────────────────────────
train = pd.read_csv("train.csv")
print("=" * 60)
print("TRAIN — Shape:", train.shape)
print("=" * 60)
print("\n── Dtypes & Missing ──")
info = pd.DataFrame({
    "dtype": train.dtypes,
    "missing": train.isnull().sum(),
    "missing_pct": (train.isnull().sum() / len(train) * 100).round(2),
    "nunique": train.nunique()
})
print(info.to_string())

print("\n── Sample (5 rows) ──")
print(train.head().to_string())

print("\n── Descriptive Stats (numerik) ──")
print(train.describe().to_string())

# ── TEST ───────────────────────────────────────────────
test = pd.read_csv("test.csv")
print("\n" + "=" * 60)
print("TEST — Shape:", test.shape)
print("=" * 60)
print(test.head().to_string())
print(test.dtypes)

# ── DATA LINGKUNGAN ────────────────────────────────────
env = pd.read_csv("data_pendukung/data_lingkungan.csv")
print("\n" + "=" * 60)
print("DATA LINGKUNGAN — Shape:", env.shape)
print("=" * 60)
info_env = pd.DataFrame({
    "dtype": env.dtypes,
    "missing": env.isnull().sum(),
    "missing_pct": (env.isnull().sum() / len(env) * 100).round(2),
    "nunique": env.nunique()
})
print(info_env.to_string())
print("\n── Sample (3 rows) ──")
print(env.head(3).to_string())

# ── KOORDINAT POS ──────────────────────────────────────
pos = pd.read_csv("data_pendukung/koordinat_pos.csv")
print("\n" + "=" * 60)
print("KOORDINAT POS — Shape:", pos.shape)
print("=" * 60)
print(pos.to_string())

TRAIN — Shape: (84396, 3)

── Dtypes & Missing ──
            dtype  missing  missing_pct  nunique
datetime      str        0          0.0     2909
nama_pos      str        0          0.0       30
tma_mdpl  float64        0          0.0    19185

── Sample (5 rows) ──
              datetime                nama_pos  tma_mdpl
0  2023-01-01 06:00:00  Arjowinangun - Pacitan      1.30
1  2023-01-01 12:00:00  Arjowinangun - Pacitan      1.20
2  2023-01-01 18:00:00  Arjowinangun - Pacitan      1.50
3  2023-01-02 06:00:00  Arjowinangun - Pacitan      1.45
4  2023-01-02 12:00:00  Arjowinangun - Pacitan      1.25

── Descriptive Stats (numerik) ──
           tma_mdpl
count  84396.000000
mean      56.482537
std       46.765914
min       -0.059668
25%       10.100000
50%       50.370000
75%       90.660938
max      325.830000

TEST — Shape: (21780, 1)
                                             id
0  2025-09-19 06:00:00 - Arjowinangun - Pacitan
1  2025-09-19 12:00:00 - Arjowinangun - Pacitan
2  2

In [3]:
import pandas as pd

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

# ── Periode train ──
train['datetime'] = pd.to_datetime(train['datetime'])
print("TRAIN periode:")
print("  Start :", train['datetime'].min())
print("  End   :", train['datetime'].max())
print("  Total rows:", len(train))

# ── Periode test ──
test['datetime'] = pd.to_datetime(test['id'].str[:19])
print("\nTEST periode:")
print("  Start :", test['datetime'].min())
print("  End   :", test['datetime'].max())
print("  Total rows:", len(test))

# ── Gap train → test ──
gap = test['datetime'].min() - train['datetime'].max()
print(f"\nGAP train→test: {gap}")

# ── 30 nama_pos ──
print("\n30 POS di train:")
for i, p in enumerate(sorted(train['nama_pos'].unique())):
    print(f"  {i+1:2d}. {p}")

# ── Apakah semua pos ada di test? ──
pos_train = set(test['id'].str[22:].unique())  # ambil nama pos dari id test
pos_train2 = set(train['nama_pos'].unique())
print("\nPos di test tapi tidak di train:", pos_train - pos_train2)
print("Pos di train tapi tidak di test:", pos_train2 - pos_train)

# ── Jumlah timestamp unik per pos di train ──
print("\nJumlah observasi per pos (train):")
print(train['nama_pos'].value_counts().to_string())

TRAIN periode:
  Start : 2023-01-01 06:00:00
  End   : 2025-09-18 18:00:00
  Total rows: 84396

TEST periode:
  Start : 2025-09-19 06:00:00
  End   : 2026-05-18 18:00:00
  Total rows: 21780

GAP train→test: 0 days 12:00:00

30 POS di train:
   1. Arjowinangun - Pacitan
   2. Babat
   3. Badegan
   4. Bengkelolor
   5. Boboh Kali Lamong
   6. Bojonegoro - Kali Kethek
   7. Brangkal
   8. Cepu
   9. Colo Weir
  10. Floodway Bridge C
  11. Gunungsari
  12. Jarum
  13. Jurug
  14. Kajangan
  15. Kali Anyar - Kreteg Abang
  16. Kali Pepe - PTPN
  17. Kali Pepe - Tugu Boto
  18. Karanggeneng
  19. Karangnongko
  20. Kedungupit
  21. Ketonggo
  22. Lorog
  23. Napel
  24. Ngadipiro
  25. Ngrembang
  26. Peren
  27. Sekayu
  28. Serenan
  29. Sumberrejo
  30. Wonogiri Dam

Pos di test tapi tidak di train: set()
Pos di train tapi tidak di test: set()

Jumlah observasi per pos (train):
nama_pos
Arjowinangun - Pacitan       2903
Colo Weir                    2903
Jurug                        2903


In [4]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")

# ── Stats per pos ──
print("Stats tma_mdpl per pos:")
stats = train.groupby('nama_pos')['tma_mdpl'].agg(['mean','std','min','max','count'])
stats = stats.sort_values('mean', ascending=False)
print(stats.round(3).to_string())

# ── Nilai negatif ──
neg = train[train['tma_mdpl'] < 0]
print(f"\nNilai tma_mdpl NEGATIF: {len(neg)} baris")
print(neg[['datetime','nama_pos','tma_mdpl']].to_string())

# ── Distribusi global ──
print("\nPercentiles tma_mdpl:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p:2d}: {np.percentile(train['tma_mdpl'], p):.3f}")

Stats tma_mdpl per pos:
                              mean    std      min      max  count
nama_pos                                                          
Ngadipiro                  143.563  0.335  143.178  146.310   2901
Ngrembang                  140.051  0.287  139.850  143.790   2901
Wonogiri Dam               132.340  3.300  125.540  137.357   2901
Badegan                    122.402  0.310  121.902  123.960   2901
Colo Weir                  107.790  1.119  102.186  109.700   2903
Kali Pepe - Tugu Boto       94.821  0.448   94.351  100.420   2900
Peren                       91.305  2.016   90.108  170.100   2902
Jarum                       90.721  3.918   89.290  250.140   2864
Sekayu                      87.273  0.659   86.607   92.320   2901
Kali Anyar - Kreteg Abang   86.461  4.464   84.444  323.207   2901
Serenan                     86.322  0.705   85.470   90.980   2901
Kali Pepe - PTPN            82.547  2.003    0.000  138.070   2901
Jurug                       78.817  1.

In [5]:
import pandas as pd

train = pd.read_csv("train.csv")

# Cek nilai 0 per pos
zero = train[train['tma_mdpl'] == 0.0]
print(f"Total nilai 0: {len(zero)}")
print("\nPer pos:")
print(zero['nama_pos'].value_counts().to_string())

print("\nSample baris nilai 0:")
print(zero[['datetime','nama_pos','tma_mdpl']].head(20).to_string())

Total nilai 0: 4

Per pos:
nama_pos
Bojonegoro - Kali Kethek    1
Jurug                       1
Kali Pepe - PTPN            1
Karanggeneng                1

Sample baris nilai 0:
                  datetime                  nama_pos  tma_mdpl
15016  2023-09-14 06:00:00  Bojonegoro - Kali Kethek       0.0
34492  2025-02-28 06:00:00                     Jurug       0.0
41825  2023-11-04 18:00:00          Kali Pepe - PTPN       0.0
47644  2023-11-10 18:00:00              Karanggeneng       0.0


In [6]:
import pandas as pd
import numpy as np

env = pd.read_csv("data_pendukung/data_lingkungan.csv")

# ── Statistik numerik semua fitur ──
num_cols = env.select_dtypes(include='number').columns.tolist()
print("── Descriptive Stats (fitur numerik) ──")
print(env[num_cols].describe().round(3).to_string())

# ── Cek nilai 0 dan negatif per kolom curah hujan ──
rain_cols = ['rainfall_mm', 'rainfall_openmeteo_mm', 'rainfall_max_24h_mm']
print("\n── Curah hujan — % nilai > 0 ──")
for c in rain_cols:
    pct = (env[c] > 0).mean() * 100
    print(f"  {c}: {pct:.2f}% non-zero")

# ── Landcover distribution ──
print("\n── Landcover class distribution ──")
print(env['landcover_name'].value_counts().to_string())

# ── built_surface_m2 per pos ──
print("\n── built_surface_m2 per pos (unik?) ──")
print(env.groupby('nama_pos')['built_surface_m2'].nunique().to_string())

# ── MJO active rate ──
print("\n── MJO active rate ──")
print(env['mjo_active'].value_counts(normalize=True).mul(100).round(2).to_string())

# ── nino_34: cek pola missing ──
print("\n── nino_34 missing — per bulan? ──")
env['datetime'] = pd.to_datetime(env['datetime'])
env['ym'] = env['datetime'].dt.to_period('M')
missing_nino = env[env['nino_34'].isna()].groupby('ym').size()
print(f"  Jumlah period dengan missing nino_34: {len(missing_nino)}")
print(missing_nino.head(10).to_string())

── Descriptive Stats (fitur numerik) ──
       rainfall_mm  humidity_pct  wind_direction_deg  dew_point_c  cloud_cover_pct  temperature_c  wind_speed_kmh  rainfall_openmeteo_mm  rainfall_max_24h_mm  solar_radiation_mj_m2  soil_moisture_0_7cm  soil_moisture_7_28cm  soil_moisture_28_100cm  soil_moisture_100_255cm  surface_pressure_hpa  pressure_msl_hpa  built_surface_m2  landcover_class        rmm1        rmm2   mjo_phase  mjo_amplitude  mjo_active     nino_34
count   888480.000    888480.000          888480.000   888480.000       888480.000     888480.000      888480.000             888480.000           888480.000             888480.000           887760.000            887760.000              887760.000               887760.000            887760.000        887760.000        888480.000       888480.000  887760.000  887760.000  887760.000     887760.000  887760.000  875520.000
mean         0.278        79.822             186.330       22.443           75.552         26.623           8.293 

In [7]:
import pandas as pd
import numpy as np

env = pd.read_csv("data_pendukung/data_lingkungan.csv")

# Cek solar radiation nilai -999
bad_solar = env[env['solar_radiation_mj_m2'] == -999]
print(f"Baris solar_radiation = -999: {len(bad_solar)}")
print(bad_solar[['datetime','nama_pos','solar_radiation_mj_m2']].head(10).to_string())

# Apakah rainfall_mm == rainfall_openmeteo_mm selalu?
diff = (env['rainfall_mm'] != env['rainfall_openmeteo_mm']).sum()
print(f"\nBaris rainfall_mm != rainfall_openmeteo_mm: {diff}")

# Missing 720 baris — pos mana dan periode mana?
env['datetime'] = pd.to_datetime(env['datetime'])
missing_soil = env[env['soil_moisture_0_7cm'].isna()]
print(f"\nMissing soil_moisture — total: {len(missing_soil)}")
print("Per pos:")
print(missing_soil['nama_pos'].value_counts().to_string())
print("\nPeriode:")
print(f"  Start: {missing_soil['datetime'].min()}")
print(f"  End  : {missing_soil['datetime'].max()}")

Baris solar_radiation = -999: 100080
                  datetime                nama_pos  solar_radiation_mj_m2
26280  2025-12-31 00:00:00  Arjowinangun - Pacitan                 -999.0
26281  2025-12-31 01:00:00  Arjowinangun - Pacitan                 -999.0
26282  2025-12-31 02:00:00  Arjowinangun - Pacitan                 -999.0
26283  2025-12-31 03:00:00  Arjowinangun - Pacitan                 -999.0
26284  2025-12-31 04:00:00  Arjowinangun - Pacitan                 -999.0
26285  2025-12-31 05:00:00  Arjowinangun - Pacitan                 -999.0
26286  2025-12-31 06:00:00  Arjowinangun - Pacitan                 -999.0
26287  2025-12-31 07:00:00  Arjowinangun - Pacitan                 -999.0
26288  2025-12-31 08:00:00  Arjowinangun - Pacitan                 -999.0
26289  2025-12-31 09:00:00  Arjowinangun - Pacitan                 -999.0

Baris rainfall_mm != rainfall_openmeteo_mm: 0

Missing soil_moisture — total: 720
Per pos:
nama_pos
Arjowinangun - Pacitan       24
Babat           

In [8]:
import pandas as pd

env = pd.read_csv("data_pendukung/data_lingkungan.csv")
env['datetime'] = pd.to_datetime(env['datetime'])

# Kapan -999 pertama muncul?
bad = env[env['solar_radiation_mj_m2'] == -999]
print(f"Pertama kali -999: {bad['datetime'].min()}")
print(f"Terakhir kali -999: {bad['datetime'].max()}")

# Berapa baris -999 di periode test (Sep 2025 – Mei 2026)?
test_period = env[env['datetime'] >= '2025-09-19']
bad_test = test_period[test_period['solar_radiation_mj_m2'] == -999]
print(f"\nTotal baris di test period: {len(test_period)}")
print(f"Baris -999 di test period : {len(bad_test)}")
print(f"Persentase               : {len(bad_test)/len(test_period)*100:.1f}%")

# Distribusi -999 per bulan
env['ym'] = env['datetime'].dt.to_period('M')
bad_per_month = env[env['solar_radiation_mj_m2'] == -999].groupby('ym').size()
print("\n-999 per bulan:")
print(bad_per_month.to_string())

Pertama kali -999: 2025-12-31 00:00:00
Terakhir kali -999: 2026-05-18 23:00:00

Total baris di test period: 174240
Baris -999 di test period : 100080
Persentase               : 57.4%

-999 per bulan:
ym
2025-12      720
2026-01    22320
2026-02    20160
2026-03    22320
2026-04    21600
2026-05    12960
Freq: M


In [9]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
env   = pd.read_csv("data_pendukung/data_lingkungan.csv")

# ── Merge train dengan env ──
# TMA diukur jam 06/12/18, env per jam → ambil jam yang sama
train['datetime'] = pd.to_datetime(train['datetime'])
env['datetime']   = pd.to_datetime(env['datetime'])

merged = train.merge(env, on=['datetime','nama_pos'], how='left')
print(f"Merged shape: {merged.shape}")
print(f"Missing setelah merge (tma_mdpl): {merged['tma_mdpl'].isna().sum()}")

# ── Korelasi fitur numerik vs tma_mdpl ──
num_cols = [
    'rainfall_mm','humidity_pct','wind_direction_deg','dew_point_c',
    'cloud_cover_pct','temperature_c','wind_speed_kmh',
    'rainfall_max_24h_mm','solar_radiation_mj_m2',
    'soil_moisture_0_7cm','soil_moisture_7_28cm',
    'soil_moisture_28_100cm','soil_moisture_100_255cm',
    'surface_pressure_hpa','pressure_msl_hpa',
    'built_surface_m2','landcover_class',
    'rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active','nino_34'
]

corr = merged[num_cols + ['tma_mdpl']].corr()['tma_mdpl'].drop('tma_mdpl')
print("\n── Korelasi vs tma_mdpl (sorted) ──")
print(corr.sort_values(ascending=False).round(4).to_string())

# ── Korelasi DALAM pos (hilangkan efek elevasi) ──
print("\n── Korelasi vs tma_mdpl WITHIN pos (demeaned) ──")
merged['tma_demeaned'] = merged.groupby('nama_pos')['tma_mdpl'].transform(lambda x: x - x.mean())
corr_within = merged[num_cols + ['tma_demeaned']].corr()['tma_demeaned'].drop('tma_demeaned')
print(corr_within.sort_values(ascending=False).round(4).to_string())

Merged shape: (84396, 28)
Missing setelah merge (tma_mdpl): 0

── Korelasi vs tma_mdpl (sorted) ──
soil_moisture_100_255cm    0.1894
built_surface_m2           0.1793
soil_moisture_28_100cm     0.1270
soil_moisture_7_28cm       0.1238
soil_moisture_0_7cm        0.1058
pressure_msl_hpa           0.0970
wind_direction_deg         0.0636
wind_speed_kmh             0.0573
nino_34                    0.0178
mjo_amplitude              0.0044
mjo_phase                  0.0039
rmm2                       0.0027
mjo_active                 0.0008
humidity_pct               0.0004
cloud_cover_pct            0.0003
rmm1                      -0.0025
rainfall_mm               -0.0053
solar_radiation_mj_m2     -0.0130
rainfall_max_24h_mm       -0.0251
temperature_c             -0.0770
dew_point_c               -0.1217
landcover_class           -0.1573
surface_pressure_hpa      -0.9474

── Korelasi vs tma_mdpl WITHIN pos (demeaned) ──
soil_moisture_7_28cm       0.2408
soil_moisture_0_7cm        0.2289
s

In [10]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
train['datetime'] = pd.to_datetime(train['datetime'])
train['month'] = train['datetime'].dt.month
train['hour'] = train['datetime'].dt.hour
train['year'] = train['datetime'].dt.year

# ── Pola musiman: TMA rata-rata per bulan ──
print("── TMA mean per bulan (semua pos) ──")
print(train.groupby('month')['tma_mdpl'].mean().round(3).to_string())

# ── Pola per jam ──
print("\n── TMA mean per jam observasi (06/12/18) ──")
print(train.groupby('hour')['tma_mdpl'].mean().round(3).to_string())

# ── Tren tahunan ──
print("\n── TMA mean per tahun ──")
print(train.groupby('year')['tma_mdpl'].mean().round(3).to_string())

# ── Pos paling dinamis: autocorrelation lag-1 ──
print("\n── Autocorrelation lag-1 per pos ──")
results = []
for pos, grp in train.groupby('nama_pos'):
    grp = grp.sort_values('datetime')
    ac = grp['tma_mdpl'].autocorr(lag=1)
    results.append({'nama_pos': pos, 'autocorr_lag1': round(ac, 4)})
ac_df = pd.DataFrame(results).sort_values('autocorr_lag1')
print(ac_df.to_string(index=False))

# ── Cek gap waktu antar observasi per pos ──
print("\n── Gap waktu antar observasi (seharusnya 6 jam) ──")
sample_pos = train[train['nama_pos'] == 'Arjowinangun - Pacitan'].sort_values('datetime')
diffs = sample_pos['datetime'].diff().dropna()
print(diffs.value_counts().head(5).to_string())

── TMA mean per bulan (semua pos) ──
month
1     57.141
2     57.804
3     57.319
4     56.589
5     56.428
6     55.980
7     56.174
8     56.236
9     55.677
10    55.577
11    56.061
12    56.781

── TMA mean per jam observasi (06/12/18) ──
hour
6     56.484
12    56.475
18    56.489

── TMA mean per tahun ──
year
2023    57.654
2024    56.011
2025    55.483

── Autocorrelation lag-1 per pos ──
                 nama_pos  autocorr_lag1
         Kali Pepe - PTPN         0.0085
                    Jarum         0.0235
Kali Anyar - Kreteg Abang         0.0364
                    Peren         0.0470
             Karangnongko         0.0779
                    Napel         0.0939
 Bojonegoro - Kali Kethek         0.1327
                 Kajangan         0.2247
               Kedungupit         0.2326
                    Jurug         0.3815
        Floodway Bridge C         0.5526
    Kali Pepe - Tugu Boto         0.6108
   Arjowinangun - Pacitan         0.6240
                Ngrembang

In [11]:
import pandas as pd

train = pd.read_csv("train.csv")
train['datetime'] = pd.to_datetime(train['datetime'])
train = train.sort_values(['nama_pos','datetime'])

# Cari semua gap > 6 jam
train['prev_dt'] = train.groupby('nama_pos')['datetime'].shift(1)
train['gap'] = train['datetime'] - train['prev_dt']

big_gaps = train[train['gap'] > pd.Timedelta('6h')].copy()
big_gaps['gap_hours'] = big_gaps['gap'].dt.total_seconds() / 3600

print(f"Total gap > 6 jam: {len(big_gaps)}")
print("\nSemua gap > 12 jam:")
large = big_gaps[big_gaps['gap_hours'] > 12].sort_values('gap_hours', ascending=False)
print(large[['nama_pos','prev_dt','datetime','gap_hours']].to_string())

Total gap > 6 jam: 28129

Semua gap > 12 jam:
                        nama_pos             prev_dt            datetime  gap_hours
26494          Floodway Bridge C 2023-08-22 06:00:00 2024-02-01 06:00:00     3912.0
14996   Bojonegoro - Kali Kethek 2023-08-03 12:00:00 2023-09-06 12:00:00      816.0
14995   Bojonegoro - Kali Kethek 2023-07-03 06:00:00 2023-08-03 12:00:00      750.0
16540   Bojonegoro - Kali Kethek 2025-02-03 18:00:00 2025-03-05 12:00:00      714.0
51901               Karangnongko 2025-02-03 18:00:00 2025-03-01 06:00:00      612.0
54802                 Kedungupit 2025-02-03 18:00:00 2025-03-01 06:00:00      612.0
10937                Bengkelolor 2025-02-03 18:00:00 2025-03-01 06:00:00      612.0
8036                     Badegan 2025-02-03 18:00:00 2025-03-01 06:00:00      612.0
19428                   Brangkal 2025-02-03 18:00:00 2025-03-01 06:00:00      612.0
5149                       Babat 2025-02-03 18:00:00 2025-03-01 06:00:00      612.0
28727                 Gunungsa

In [12]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
env   = pd.read_csv("data_pendukung/data_lingkungan.csv")
train['datetime'] = pd.to_datetime(train['datetime'])
env['datetime']   = pd.to_datetime(env['datetime'])

# ── Duplicate rows ──
print(f"Duplikat di train: {train.duplicated().sum()}")
print(f"Duplikat (datetime+pos): {train.duplicated(['datetime','nama_pos']).sum()}")
print(f"Duplikat di env: {env.duplicated().sum()}")
print(f"Duplikat (datetime+pos) env: {env.duplicated(['datetime','nama_pos']).sum()}")

# ── Train vs test period: apakah env menutupi test? ──
test = pd.read_csv("test.csv")
test['datetime'] = pd.to_datetime(test['id'].str[:19])
print(f"\nTest datetime range : {test['datetime'].min()} → {test['datetime'].max()}")
print(f"Env datetime range  : {env['datetime'].min()} → {env['datetime'].max()}")

# ── Distribusi TMA per bulan per pos (cek seasonality within-pos) ──
train['month'] = train['datetime'].dt.month
seasonal = train.groupby(['nama_pos','month'])['tma_mdpl'].mean().unstack()
# Cari pos dengan variasi musiman tinggi
seasonal['range'] = seasonal.max(axis=1) - seasonal.min(axis=1)
print("\n── Variasi musiman TMA per pos (max-min across months) ──")
print(seasonal['range'].sort_values(ascending=False).round(3).to_string())

# ── Outlier ekstrem per pos (> mean + 5*std) ──
print("\n── Outlier ekstrem per pos (> mean + 5*std) ──")
for pos, grp in train.groupby('nama_pos'):
    mu, sd = grp['tma_mdpl'].mean(), grp['tma_mdpl'].std()
    outliers = grp[grp['tma_mdpl'] > mu + 5*sd]
    if len(outliers) > 0:
        print(f"  {pos}: {len(outliers)} outlier | max={grp['tma_mdpl'].max():.2f} | threshold={mu+5*sd:.2f}")

Duplikat di train: 0
Duplikat (datetime+pos): 0
Duplikat di env: 0
Duplikat (datetime+pos) env: 0

Test datetime range : 2025-09-19 06:00:00 → 2026-05-18 18:00:00
Env datetime range  : 2023-01-01 00:00:00 → 2026-05-18 23:00:00

── Variasi musiman TMA per pos (max-min across months) ──
nama_pos
Wonogiri Dam                 8.763
Bojonegoro - Kali Kethek     4.885
Ketonggo                     3.713
Napel                        3.672
Bengkelolor                  3.531
Karangnongko                 3.277
Boboh Kali Lamong            3.254
Kedungupit                   3.180
Cepu                         3.133
Colo Weir                    2.956
Sumberrejo                   2.908
Kajangan                     2.751
Jurug                        2.345
Karanggeneng                 2.295
Jarum                        1.980
Gunungsari                   1.968
Floodway Bridge C            1.952
Brangkal                     1.695
Kali Anyar - Kreteg Abang    1.659
Serenan                      1.464
Babat

In [13]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv", parse_dates=["datetime"])

stats = train.groupby("nama_pos")["tma_mdpl"].agg(
    mean="mean",
    std="std",
    min="min",
    max="max",
    skew=lambda x: x.skew(),
    cv=lambda x: x.std() / x.mean() if x.mean() != 0 else np.nan
).round(3)

stats["range"] = stats["max"] - stats["min"]
stats = stats.sort_values("mean", ascending=False)
print(stats.to_string())

                              mean    std      min      max    skew     cv    range
nama_pos                                                                           
Ngadipiro                  143.563  0.335  143.178  146.310   2.346  0.002    3.132
Ngrembang                  140.051  0.287  139.850  143.790   5.410  0.002    3.940
Wonogiri Dam               132.340  3.300  125.540  137.357  -0.498  0.025   11.817
Badegan                    122.402  0.310  121.902  123.960   1.224  0.003    2.058
Colo Weir                  107.790  1.119  102.186  109.700  -3.158  0.010    7.514
Kali Pepe - Tugu Boto       94.821  0.448   94.351  100.420   3.905  0.005    6.069
Peren                       91.305  2.016   90.108  170.100  34.316  0.022   79.992
Jarum                       90.721  3.918   89.290  250.140  32.935  0.043  160.850
Sekayu                      87.273  0.659   86.607   92.320   2.483  0.008    5.713
Kali Anyar - Kreteg Abang   86.461  4.464   84.444  323.207  51.463  0.052  

In [14]:
import pandas as pd
import numpy as np

env = pd.read_csv("data_pendukung/data_lingkungan.csv", parse_dates=["datetime"])

# Pisahkan train dan test period
train_end = pd.Timestamp("2025-09-18 18:00:00")
test_start = pd.Timestamp("2025-09-19 06:00:00")

env_train = env[env["datetime"] <= train_end]
env_test  = env[env["datetime"] >= test_start]

# Replace sentinel -999 dulu
env_train_clean = env_train.replace(-999, np.nan)
env_test_clean  = env_test.replace(-999, np.nan)

cols = [c for c in env.columns if c not in ["datetime", "nama_pos"]]

missing = pd.DataFrame({
    "train_missing_pct": (env_train_clean[cols].isna().mean() * 100).round(2),
    "test_missing_pct":  (env_test_clean[cols].isna().mean() * 100).round(2),
})
missing["delta"] = (missing["test_missing_pct"] - missing["train_missing_pct"]).round(2)
missing = missing.sort_values("test_missing_pct", ascending=False)

print(missing.to_string())
print(f"\nenv_train rows: {len(env_train):,}")
print(f"env_test rows:  {len(env_test):,}")

                         train_missing_pct  test_missing_pct  delta
solar_radiation_mj_m2                  0.0             57.50  57.50
nino_34                                0.0              7.45   7.45
pressure_msl_hpa                       0.0              0.41   0.41
soil_moisture_28_100cm                 0.0              0.41   0.41
soil_moisture_100_255cm                0.0              0.41   0.41
soil_moisture_0_7cm                    0.0              0.41   0.41
soil_moisture_7_28cm                   0.0              0.41   0.41
surface_pressure_hpa                   0.0              0.41   0.41
mjo_amplitude                          0.0              0.41   0.41
mjo_active                             0.0              0.41   0.41
rmm2                                   0.0              0.41   0.41
mjo_phase                              0.0              0.41   0.41
rmm1                                   0.0              0.41   0.41
rainfall_max_24h_mm                    0.0      

In [15]:
import pandas as pd
import numpy as np

env = pd.read_csv("data_pendukung/data_lingkungan.csv", parse_dates=["datetime"])

train_end = pd.Timestamp("2025-09-18 18:00:00")
env_train = env[env["datetime"] <= train_end].replace(-999, np.nan)

num_cols = [
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "wind_direction_deg", "dew_point_c",
    "cloud_cover_pct", "temperature_c", "wind_speed_kmh",
    "solar_radiation_mj_m2",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm",
    "soil_moisture_28_100cm", "soil_moisture_100_255cm",
    "surface_pressure_hpa", "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_phase", "mjo_amplitude", "mjo_active",
    "nino_34"
]

corr = env_train[num_cols].corr().round(3)

# Tampilkan pair dengan |corr| > 0.7 (multicollinearity zone)
pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) > 0.7:
            pairs.append({
                "feature_a": corr.columns[i],
                "feature_b": corr.columns[j],
                "corr": val
            })

pairs_df = pd.DataFrame(pairs).sort_values("corr", key=abs, ascending=False)
print("=== Pairs dengan |corr| > 0.7 ===")
print(pairs_df.to_string(index=False))

# Full corr matrix untuk soil moisture dan pressure
focus_cols = [
    "soil_moisture_0_7cm", "soil_moisture_7_28cm",
    "soil_moisture_28_100cm", "soil_moisture_100_255cm",
    "surface_pressure_hpa", "pressure_msl_hpa"
]
print("\n=== Corr Matrix: Soil Moisture + Pressure ===")
print(corr.loc[focus_cols, focus_cols].to_string())

=== Pairs dengan |corr| > 0.7 ===
           feature_a              feature_b   corr
 soil_moisture_0_7cm   soil_moisture_7_28cm  0.914
        humidity_pct          temperature_c -0.862
soil_moisture_7_28cm soil_moisture_28_100cm  0.825
       mjo_amplitude             mjo_active  0.730
       temperature_c  solar_radiation_mj_m2  0.716

=== Corr Matrix: Soil Moisture + Pressure ===
                         soil_moisture_0_7cm  soil_moisture_7_28cm  soil_moisture_28_100cm  soil_moisture_100_255cm  surface_pressure_hpa  pressure_msl_hpa
soil_moisture_0_7cm                    1.000                 0.914                   0.683                    0.356                -0.224            -0.341
soil_moisture_7_28cm                   0.914                 1.000                   0.825                    0.452                -0.239            -0.329
soil_moisture_28_100cm                 0.683                 0.825                   1.000                    0.652                -0.200        

In [16]:
import pandas as pd
import numpy as np
from scipy import stats

env = pd.read_csv("data_pendukung/data_lingkungan.csv", parse_dates=["datetime"])

train_end = pd.Timestamp("2025-09-18 18:00:00")
test_start = pd.Timestamp("2025-09-19 06:00:00")

env_train = env[env["datetime"] <= train_end].replace(-999, np.nan)
env_test  = env[env["datetime"] >= test_start].replace(-999, np.nan)

check_cols = [
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c",
    "cloud_cover_pct", "temperature_c", "wind_speed_kmh",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm",
    "soil_moisture_28_100cm",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "nino_34"
]

rows = []
for col in check_cols:
    tr = env_train[col].dropna()
    te = env_test[col].dropna()
    if len(tr) == 0 or len(te) == 0:
        continue
    ks_stat, ks_p = stats.ks_2samp(
        tr.sample(min(10000, len(tr)), random_state=42),
        te.sample(min(10000, len(te)), random_state=42)
    )
    rows.append({
        "feature": col,
        "train_mean": round(tr.mean(), 4),
        "test_mean":  round(te.mean(), 4),
        "mean_delta_pct": round((te.mean() - tr.mean()) / (abs(tr.mean()) + 1e-9) * 100, 2),
        "train_std": round(tr.std(), 4),
        "test_std":  round(te.std(), 4),
        "ks_stat": round(ks_stat, 4),
        "ks_p": round(ks_p, 6)
    })

drift_df = pd.DataFrame(rows).sort_values("ks_stat", ascending=False)
print(drift_df.to_string(index=False))

               feature  train_mean  test_mean  mean_delta_pct  train_std  test_std  ks_stat  ks_p
               nino_34      0.3633    -0.3653         -200.56     0.8434    0.3217   0.5080   0.0
  soil_moisture_7_28cm      0.3453     0.4054           17.40     0.0998    0.0776   0.3107   0.0
   soil_moisture_0_7cm      0.3471     0.4020           15.82     0.1088    0.0855   0.2799   0.0
soil_moisture_28_100cm      0.3443     0.3846           11.72     0.0858    0.0785   0.2354   0.0
           dew_point_c     22.2853    23.0910            3.62     2.1059    1.1183   0.2176   0.0
   rainfall_max_24h_mm      2.6697     3.5708           33.75     3.7916    3.8629   0.2006   0.0
       cloud_cover_pct     73.2300    85.0823           16.19    35.5398   27.8968   0.1811   0.0
                  rmm2     -0.0137     0.2395         1853.83     1.0465    1.0790   0.1807   0.0
      pressure_msl_hpa   1010.4029  1009.5900           -0.08     2.0062    1.8528   0.1616   0.0
           rainfall_

In [17]:
import pandas as pd
import numpy as np

env = pd.read_csv("data_pendukung/data_lingkungan.csv", parse_dates=["datetime"])

train_end = pd.Timestamp("2025-09-18 18:00:00")
test_start = pd.Timestamp("2025-09-19 06:00:00")

env_train = env[env["datetime"] <= train_end].replace(-999, np.nan)
env_test  = env[env["datetime"] >= test_start].replace(-999, np.nan)

check_cols = [
    "soil_moisture_0_7cm", "soil_moisture_7_28cm",
    "soil_moisture_28_100cm", "rainfall_mm",
    "rainfall_max_24h_mm", "nino_34"
]

rows = []
for col in check_cols:
    tr = env_train[col].dropna()
    te = env_test[col].dropna()
    tr_min, tr_max = tr.min(), tr.max()
    te_min, te_max = te.min(), te.max()
    oor_pct = ((te < tr_min) | (te > tr_max)).mean() * 100
    rows.append({
        "feature": col,
        "train_min": round(tr_min, 4),
        "train_max": round(tr_max, 4),
        "test_min":  round(te_min, 4),
        "test_max":  round(te_max, 4),
        "test_OOR_pct": round(oor_pct, 2)
    })

oor_df = pd.DataFrame(rows)
print(oor_df.to_string(index=False))

               feature  train_min  train_max  test_min  test_max  test_OOR_pct
   soil_moisture_0_7cm      0.108       0.52     0.089     0.541          0.09
  soil_moisture_7_28cm      0.118       0.52     0.161     0.521          0.00
soil_moisture_28_100cm      0.135       0.52     0.151     0.520          0.00
           rainfall_mm      0.000      34.60     0.000    40.500          0.00
   rainfall_max_24h_mm      0.000      34.60     0.000    40.500          0.01
               nino_34     -0.740       2.03    -0.700     0.230          0.00


In [18]:
import pandas as pd

train = pd.read_csv("train.csv", parse_dates=["datetime"])

# Cek per pos: cari gap terbesar di Feb-Mar 2025
target_window = train[
    (train["datetime"] >= "2025-01-15") &
    (train["datetime"] <= "2025-03-15")
].copy()

# Ambil satu pos representatif (Wonogiri Dam — data paling lengkap)
wonogiri = target_window[
    target_window["nama_pos"] == "Wonogiri Dam"
].sort_values("datetime")

print("=== Wonogiri Dam — Jan 15 s/d Mar 15 2025 ===")
print(wonogiri[["datetime", "tma_mdpl"]].to_string(index=False))

=== Wonogiri Dam — Jan 15 s/d Mar 15 2025 ===
           datetime   tma_mdpl
2025-01-15 06:00:00 133.940000
2025-01-15 12:00:00 133.940000
2025-01-15 18:00:00 133.940000
2025-01-16 06:00:00 133.920000
2025-01-16 12:00:00 133.920000
2025-01-16 18:00:00 133.910000
2025-01-17 06:00:00 133.900000
2025-01-17 12:00:00 133.880000
2025-01-17 18:00:00 133.880000
2025-01-18 06:00:00 133.860000
2025-01-18 12:00:00 133.850000
2025-01-18 18:00:00 133.850000
2025-01-19 06:00:00 133.840000
2025-01-19 12:00:00 133.830000
2025-01-19 18:00:00 133.830000
2025-01-20 06:00:00 133.433886
2025-01-20 12:00:00 133.880000
2025-01-20 18:00:00 133.920000
2025-01-21 06:00:00 133.940000
2025-01-21 12:00:00 133.950000
2025-01-21 18:00:00 134.030000
2025-01-22 06:00:00 134.340000
2025-01-22 12:00:00 135.045242
2025-01-22 18:00:00 134.480000
2025-01-23 06:00:00 134.560000
2025-01-23 12:00:00 134.630000
2025-01-23 18:00:00 134.670000
2025-01-24 06:00:00 134.690000
2025-01-24 12:00:00 134.596651
2025-01-24 18:00:00 134.

In [19]:
import pandas as pd

env = pd.read_csv("data_pendukung/data_lingkungan.csv", parse_dates=["datetime"])

# Ambil satu pos, lihat apakah rainfall_max_24h_mm pada jam T
# lebih besar dari rainfall_mm pada jam T saja
# (kalau centered, nilai jam T bisa di-include)

sample = env[env["nama_pos"] == "Wonogiri Dam"].sort_values("datetime").head(72)
print(sample[["datetime", "rainfall_mm", "rainfall_max_24h_mm"]].to_string(index=False))

           datetime  rainfall_mm  rainfall_max_24h_mm
2023-01-01 00:00:00          0.0                  0.0
2023-01-01 01:00:00          0.0                  0.0
2023-01-01 02:00:00          0.0                  0.0
2023-01-01 03:00:00          0.0                  0.0
2023-01-01 04:00:00          0.0                  0.0
2023-01-01 05:00:00          0.0                  0.0
2023-01-01 06:00:00          0.0                  0.0
2023-01-01 07:00:00          0.0                  0.0
2023-01-01 08:00:00          0.0                  0.0
2023-01-01 09:00:00          0.0                  0.0
2023-01-01 10:00:00          0.1                  0.1
2023-01-01 11:00:00          0.2                  0.2
2023-01-01 12:00:00          6.2                  6.2
2023-01-01 13:00:00          0.1                  6.2
2023-01-01 14:00:00          0.1                  6.2
2023-01-01 15:00:00          3.7                  6.2
2023-01-01 16:00:00          0.8                  6.2
2023-01-01 17:00:00         

In [20]:
import pandas as pd
import numpy as np

# ── 0. LOAD RAW DATA ──────────────────────────────────────────────────────────
print("Loading data...")
train = pd.read_csv("train.csv", parse_dates=["datetime"])
test  = pd.read_csv("test.csv")
env   = pd.read_csv("data_pendukung/data_lingkungan.csv", parse_dates=["datetime"])
coords = pd.read_csv("data_pendukung/koordinat_pos.csv")

# ── 1. PARSE TEST ID ──────────────────────────────────────────────────────────
# Format: "2025-09-19 06:00:00 - Nama Pos"
test["datetime"] = pd.to_datetime(test["id"].str[:19])
test["nama_pos"] = test["id"].str[22:]

# ── 2. CLEAN ENV ──────────────────────────────────────────────────────────────
# Replace sentinel
env = env.replace(-999, np.nan)

# Drop kolom yang sudah diputuskan
drop_cols = [
    "rainfall_openmeteo_mm",
    "solar_radiation_mj_m2",
    "surface_pressure_hpa",
    "soil_moisture_100_255cm",
    "mjo_active",
    "built_surface_m2",
    "landcover_class",
    "landcover_name",
]
env = env.drop(columns=[c for c in drop_cols if c in env.columns])

# Forward-fill per pos untuk cutoff artifacts dan nino_34
env = env.sort_values(["nama_pos", "datetime"])
ff_cols = [
    "nino_34", "pressure_msl_hpa",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rmm1", "rmm2", "mjo_phase", "mjo_amplitude",
]
env[ff_cols] = env.groupby("nama_pos")[ff_cols].transform(
    lambda x: x.ffill()
)

# ── 3. ENGINEERED ROLLING FEATURES (di env, per jam) ─────────────────────────
GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")

env = env.sort_values(["nama_pos", "datetime"])

for pos, grp in env.groupby("nama_pos", sort=False):
    idx = grp.index
    rain = grp["rainfall_mm"]

    # Rolling sum backward-only
    r48 = rain.rolling(48,  min_periods=1).sum()
    r72 = rain.rolling(72,  min_periods=1).sum()
    r7d = rain.rolling(168, min_periods=1).sum()

    env.loc[idx, "rolling_rain_48h"] = r48.values
    env.loc[idx, "rolling_rain_72h"] = r72.values
    env.loc[idx, "rolling_rain_7d"]  = r7d.values

# Gap masking — set corrupt rows ke NaN
windows = {"rolling_rain_48h": 48, "rolling_rain_72h": 72, "rolling_rain_7d": 168}
for col, w in windows.items():
    corrupt_mask = (
        (env["datetime"] >= GAP_START) &
        (env["datetime"] <= GAP_END + pd.Timedelta(hours=w))
    )
    env.loc[corrupt_mask, col] = np.nan

# ── 4. MERGE TMA → ENV ───────────────────────────────────────────────────────
# TMA timestamp 06:00/12:00/18:00 — merge ke env jam yang sama
print("Merging train with env...")
train_fe = train.merge(
    env,
    on=["datetime", "nama_pos"],
    how="left"
)

print("Merging test with env...")
test_fe = test.merge(
    env,
    on=["datetime", "nama_pos"],
    how="left"
)

# ── 5. MERGE KOORDINAT ───────────────────────────────────────────────────────
train_fe = train_fe.merge(coords, on="nama_pos", how="left")
test_fe  = test_fe.merge(coords, on="nama_pos", how="left")

# ── 6. ANOMALI TRAIN — impute 5 baris (4 zero + 1 negatif) ──────────────────
# Set ke NaN dulu, lalu forward-fill per pos
anomaly_mask = train_fe["tma_mdpl"] <= 0
print(f"Anomaly rows: {anomaly_mask.sum()}")
train_fe.loc[anomaly_mask, "tma_mdpl"] = np.nan
train_fe = train_fe.sort_values(["nama_pos", "datetime"])
train_fe["tma_mdpl"] = train_fe.groupby("nama_pos")["tma_mdpl"].transform(
    lambda x: x.ffill().bfill()
)

# ── 7. TEMPORAL FEATURES ─────────────────────────────────────────────────────
for df in [train_fe, test_fe]:
    df["bulan"]       = df["datetime"].dt.month
    df["day_of_year"] = df["datetime"].dt.dayofyear
    df["hour"]        = df["datetime"].dt.hour

# ── 8. HORIZON FEATURES ──────────────────────────────────────────────────────
TRAIN_CUTOFF = pd.Timestamp("2025-09-18 18:00:00")

train_fe["horizon_days"]   = 0.0
train_fe["horizon_bucket"] = "train"

test_fe["horizon_days"] = (
    (test_fe["datetime"] - TRAIN_CUTOFF).dt.total_seconds() / 86400
).round(2)

def bucket(d):
    if d <= 30:  return "near"
    elif d <= 90: return "mid"
    else:         return "far"

test_fe["horizon_bucket"] = test_fe["horizon_days"].apply(bucket)

# ── 9. DAYS SINCE LAST VALID TMA (per pos) ───────────────────────────────────
# Untuk training: 0 (TMA tersedia)
# Untuk test: hari dari last training obs per pos

last_train_dt = (
    train.groupby("nama_pos")["datetime"].max()
    .reset_index()
    .rename(columns={"datetime": "last_train_dt"})
)
test_fe = test_fe.merge(last_train_dt, on="nama_pos", how="left")
test_fe["days_since_last_valid_tma"] = (
    (test_fe["datetime"] - test_fe["last_train_dt"]).dt.total_seconds() / 86400
).round(2)
test_fe = test_fe.drop(columns=["last_train_dt"])

train_fe["days_since_last_valid_tma"] = 0.0

# ── 10. POS STATIC STATS (per-fold di CV, global di sini untuk test) ─────────
# Hitung dari full train — dipakai untuk test inference
# Di CV loop nanti, ini dihitung ulang dari train fold only

pos_stats = train_fe.groupby("nama_pos")["tma_mdpl"].agg(
    tma_mean_pos="mean",
    tma_std_pos="std"
).reset_index()

# AC lag-1 per pos
def ac_lag1(x):
    x = x.sort_values("datetime")["tma_mdpl"].dropna()
    if len(x) < 10:
        return np.nan
    return x.autocorr(lag=1)

ac_df = train_fe.groupby("nama_pos").apply(ac_lag1).reset_index()
ac_df.columns = ["nama_pos", "ac_lag1_pos"]

pos_stats = pos_stats.merge(ac_df, on="nama_pos", how="left")

# Merge ke train dan test
train_fe = train_fe.merge(pos_stats, on="nama_pos", how="left")
test_fe  = test_fe.merge(pos_stats, on="nama_pos", how="left")

# ── 11. FINAL FEATURE LIST ───────────────────────────────────────────────────
FEATURES = [
    # Temporal
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    # Identity
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    # Exogenous
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    # Engineered
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    # Horizon
    "horizon_days", "horizon_bucket",
]

TARGET = "tma_mdpl"

# ── 12. SANITY CHECK ─────────────────────────────────────────────────────────
print("\n=== SANITY CHECK ===")
print(f"train_fe shape : {train_fe.shape}")
print(f"test_fe shape  : {test_fe.shape}")
print(f"\nFeatures missing in train_fe:")
missing_train = train_fe[FEATURES].isna().mean().round(4)
print(missing_train[missing_train > 0].to_string())
print(f"\nFeatures missing in test_fe:")
missing_test = test_fe[FEATURES].isna().mean().round(4)
print(missing_test[missing_test > 0].to_string())
print(f"\nhorizon_bucket distribution (test):")
print(test_fe["horizon_bucket"].value_counts())
print(f"\nAnomaly rows remaining in train target:")
print((train_fe["tma_mdpl"] <= 0).sum())

# ── 13. SAVE ─────────────────────────────────────────────────────────────────
# ── 13. SAVE ─────────────────────────────────────────────────────────────────
try:
    train_fe.to_parquet("train_fe.parquet", index=False, engine='fastparquet')
    test_fe.to_parquet("test_fe.parquet", index=False, engine='fastparquet')
except ImportError:
    # fallback: jika fastparquet tidak ada, gunakan pyarrow dengan reset registry
    import pyarrow
    try:
        pyarrow.unregister_extension_type('pandas.period')
    except:
        pass
    train_fe.to_parquet("train_fe.parquet", index=False, engine='pyarrow')
    test_fe.to_parquet("test_fe.parquet", index=False, engine='pyarrow')
print("\nSaved: train_fe.parquet, test_fe.parquet")
print("Done.")

Loading data...
Merging train with env...
Merging test with env...
Anomaly rows: 5

=== SANITY CHECK ===
train_fe shape : (84396, 34)
test_fe shape  : (21780, 34)

Features missing in train_fe:
rolling_rain_48h    0.0029
rolling_rain_72h    0.0039
rolling_rain_7d     0.0081

Features missing in test_fe:
Series([], )

horizon_bucket distribution (test):
horizon_bucket
far     13680
mid      5400
near     2700
Name: count, dtype: int64

Anomaly rows remaining in train target:
0

Saved: train_fe.parquet, test_fe.parquet
Done.


In [21]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

# ── LOAD ──────────────────────────────────────────────────────────────────────
train_fe = pd.read_parquet("train_fe.parquet")
test_fe  = pd.read_parquet("test_fe.parquet")

FEATURES = [
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    "horizon_days", "horizon_bucket",
]
TARGET = "tma_mdpl"
CAT_FEATURES = ["nama_pos", "horizon_bucket"]

# ── FOLD DEFINITIONS ──────────────────────────────────────────────────────────
FOLDS = [
    ("2024-07-31 18:00:00", "2024-08-01 06:00:00", "2024-10-31 18:00:00"),
    ("2024-10-31 18:00:00", "2024-11-01 06:00:00", "2025-01-31 18:00:00"),
    ("2025-01-31 18:00:00", "2025-03-01 06:00:00", "2025-06-30 18:00:00"),
    ("2025-06-30 18:00:00", "2025-07-01 06:00:00", "2025-09-18 18:00:00"),
]

GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")

ROLLING_COLS = {
    "rolling_rain_48h": 48,
    "rolling_rain_72h": 72,
    "rolling_rain_7d":  168,
}

LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.05,
    "num_leaves":       127,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     1,
    "lambda_l1":        0.1,
    "lambda_l2":        0.1,
    "verbose":          -1,
    "seed":             42,
    "deterministic":    True,
}

# ── CV LOOP ───────────────────────────────────────────────────────────────────
oof_preds = np.zeros(len(train_fe))
oof_mask  = np.zeros(len(train_fe), dtype=bool)

best_iterations = []

train_fe = train_fe.sort_values("datetime").reset_index(drop=True)

for fold_idx, (train_cut, val_start, val_end) in enumerate(FOLDS):
    train_cut = pd.Timestamp(train_cut)
    val_start = pd.Timestamp(val_start)
    val_end   = pd.Timestamp(val_end)

    tr_mask  = train_fe["datetime"] <= train_cut
    val_mask = (train_fe["datetime"] >= val_start) & (train_fe["datetime"] <= val_end)

    X_tr  = train_fe.loc[tr_mask,  FEATURES].copy()
    y_tr  = train_fe.loc[tr_mask,  TARGET]
    X_val = train_fe.loc[val_mask, FEATURES].copy()
    y_val = train_fe.loc[val_mask, TARGET]

    # Per-fold pos stats (leakage-safe)
    fold_train_df = train_fe.loc[tr_mask].copy()
    pos_stats_fold = fold_train_df.groupby("nama_pos")[TARGET].agg(
        tma_mean_pos="mean",
        tma_std_pos="std"
    )
    ac_fold = fold_train_df.groupby("nama_pos").apply(
        lambda g: g.sort_values("datetime")[TARGET].dropna().autocorr(lag=1)
    ).rename("ac_lag1_pos")
    pos_stats_fold = pos_stats_fold.join(ac_fold)

    for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
        X_tr[col]  = X_tr.index.map(
            train_fe.loc[tr_mask, "nama_pos"].map(pos_stats_fold[col])
        )
        X_val[col] = X_val.index.map(
            train_fe.loc[val_mask, "nama_pos"].map(pos_stats_fold[col])
        )

    # Mask corrupt rolling rows in val (Fold 3 only — gap overlap)
    for col, w in ROLLING_COLS.items():
        corrupt = (
            (train_fe.loc[val_mask, "datetime"] >= GAP_START) &
            (train_fe.loc[val_mask, "datetime"] <= GAP_END + pd.Timedelta(hours=w))
        )
        X_val.loc[corrupt[corrupt].index, col] = np.nan

    # Encode categoricals
    for col in CAT_FEATURES:
        X_tr[col]  = X_tr[col].astype("category")
        X_val[col] = X_val[col].astype("category")

    ds_tr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=CAT_FEATURES, free_raw_data=False)
    ds_val = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_FEATURES, free_raw_data=False)

    model = lgb.train(
        LGB_PARAMS,
        ds_tr,
        num_boost_round=2000,
        valid_sets=[ds_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iterations.append(model.best_iteration)
    preds = model.predict(X_val)
    oof_preds[val_mask] = preds
    oof_mask[val_mask]  = True

    # PERBAIKAN: gunakan np.sqrt(mean_squared_error) sebagai pengganti squared=False
    fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
    print(f"Fold {fold_idx+1} RMSE: {fold_rmse:.4f} | best_iter: {model.best_iteration}")

# ── OOF RMSE ─────────────────────────────────────────────────────────────────
oof_true = train_fe.loc[oof_mask, TARGET].values
oof_pred = oof_preds[oof_mask]
# PERBAIKAN:
oof_rmse = np.sqrt(mean_squared_error(oof_true, oof_pred))
print(f"\n=== OOF RMSE (Slot 1): {oof_rmse:.4f} ===")

# Per-pos RMSE
pos_col = train_fe.loc[oof_mask, "nama_pos"].values
rmse_per_pos = {}
for pos in np.unique(pos_col):
    mask = pos_col == pos
    # PERBAIKAN:
    rmse_per_pos[pos] = np.sqrt(mean_squared_error(oof_true[mask], oof_pred[mask]))
rmse_pos_df = pd.DataFrame.from_dict(
    rmse_per_pos, orient="index", columns=["rmse"]
).sort_values("rmse", ascending=False)
print("\n=== RMSE per Pos (top 10 worst) ===")
print(rmse_pos_df.head(10).round(4).to_string())

# ── RETRAIN FULL + INFERENCE ──────────────────────────────────────────────────
mean_best_iter = int(np.mean(best_iterations) * 1.1)  # +10% karena full data
print(f"\nRetraining full data, num_boost_round={mean_best_iter}...")

X_full = train_fe[FEATURES].copy()
y_full = train_fe[TARGET]
for col in CAT_FEATURES:
    X_full[col] = X_full[col].astype("category")

ds_full = lgb.Dataset(X_full, label=y_full, categorical_feature=CAT_FEATURES)

final_model = lgb.train(
    LGB_PARAMS,
    ds_full,
    num_boost_round=mean_best_iter,
)

# Save model
final_model.save_model(f"model_s9_exp001_cv{oof_rmse:.4f}.txt")
np.save(f"oof_exp001.npy", oof_preds)

# Inference
X_test = test_fe[FEATURES].copy()
for col in CAT_FEATURES:
    X_test[col] = X_test[col].astype("category")

test_preds = final_model.predict(X_test)

# ── SUBMISSION ────────────────────────────────────────────────────────────────
sub = pd.read_csv("sample_submission.csv")
sub["tma_mdpl"] = test_preds

# Sanity check submission
print(f"\n=== SUBMISSION SANITY CHECK ===")
print(f"Rows         : {len(sub)} (expected 21780)")
print(f"NaN count    : {sub['tma_mdpl'].isna().sum()}")
print(f"Inf count    : {np.isinf(sub['tma_mdpl']).sum()}")
print(f"Min pred     : {sub['tma_mdpl'].min():.4f}")
print(f"Max pred     : {sub['tma_mdpl'].max():.4f}")
print(f"Mean pred    : {sub['tma_mdpl'].mean():.4f}")
print(f"Train target mean : {train_fe[TARGET].mean():.4f}")

sub.to_csv(f"sub_exp001_slot1.csv", index=False)
print(f"\nSaved: sub_exp001_slot1.csv")
print(f"Model: model_s9_exp001_cv{oof_rmse:.4f}.txt")

Fold 1 RMSE: 1.5109 | best_iter: 89
Fold 2 RMSE: 1.2580 | best_iter: 143
Fold 3 RMSE: 1.3519 | best_iter: 111
Fold 4 RMSE: 0.8602 | best_iter: 104

=== OOF RMSE (Slot 1): 1.2852 ===

=== RMSE per Pos (top 10 worst) ===
                            rmse
Gunungsari                3.1469
Peren                     2.5076
Wonogiri Dam              2.0993
Floodway Bridge C         1.8536
Bojonegoro - Kali Kethek  1.8116
Karangnongko              1.7051
Napel                     1.5414
Ketonggo                  1.3347
Bengkelolor               1.2065
Kedungupit                1.1496

Retraining full data, num_boost_round=122...

=== SUBMISSION SANITY CHECK ===
Rows         : 21780 (expected 21780)
NaN count    : 0
Inf count    : 0
Min pred     : 1.0454
Max pred     : 144.4101
Mean pred    : 55.4624
Train target mean : 56.4850

Saved: sub_exp001_slot1.csv
Model: model_s9_exp001_cv1.2852.txt


In [22]:
# ── TAMBAHAN FITUR SLOT 2 (jalankan setelah load train_fe, test_fe) ───────────

# 1. tma_last_known — TMA observasi terakhir per pos di training
last_tma = (
    train_fe.sort_values("datetime")
    .groupby("nama_pos")["tma_mdpl"]
    .last()
    .reset_index()
    .rename(columns={"tma_mdpl": "tma_last_known"})
)
train_fe = train_fe.merge(last_tma, on="nama_pos", how="left")
test_fe  = test_fe.merge(last_tma, on="nama_pos", how="left")

# 2. tma_lag1 — TMA 8 jam sebelumnya (per pos, hanya valid di training)
train_fe = train_fe.sort_values(["nama_pos", "datetime"])
train_fe["tma_lag1"] = train_fe.groupby("nama_pos")["tma_mdpl"].shift(1)

# Gap masking untuk tma_lag1
GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")
corrupt_lag = (
    (train_fe["datetime"] > GAP_START) &
    (train_fe["datetime"] <= GAP_END + pd.Timedelta(hours=8))
)
train_fe.loc[corrupt_lag, "tma_lag1"] = np.nan

# Di test: set NaN (tidak tersedia)
test_fe["tma_lag1"] = np.nan

# 3. tma_rolling_mean_7d — rolling mean TMA 7 hari ke belakang (training only)
train_fe["tma_rolling_mean_7d"] = train_fe.groupby("nama_pos")["tma_mdpl"].transform(
    lambda x: x.shift(1).rolling(21, min_periods=3).mean()  # 21 obs = 7 hari × 3 obs/hari
)

# Gap masking untuk rolling mean
corrupt_roll = (
    (train_fe["datetime"] > GAP_START) &
    (train_fe["datetime"] <= GAP_END + pd.Timedelta(hours=168))
)
train_fe.loc[corrupt_roll, "tma_rolling_mean_7d"] = np.nan

# Di test: set NaN
test_fe["tma_rolling_mean_7d"] = np.nan

print("=== Slot 2 feature check ===")
print(f"tma_last_known missing train : {train_fe['tma_last_known'].isna().sum()}")
print(f"tma_last_known missing test  : {test_fe['tma_last_known'].isna().sum()}")
print(f"tma_lag1 missing train       : {train_fe['tma_lag1'].isna().mean():.4f}")
print(f"tma_lag1 missing test        : {test_fe['tma_lag1'].isna().mean():.4f}")
print(f"tma_rolling_mean_7d missing train : {train_fe['tma_rolling_mean_7d'].isna().mean():.4f}")
print(f"tma_rolling_mean_7d missing test  : {test_fe['tma_rolling_mean_7d'].isna().mean():.4f}")

=== Slot 2 feature check ===
tma_last_known missing train : 0
tma_last_known missing test  : 0
tma_lag1 missing train       : 0.0012
tma_lag1 missing test        : 1.0000
tma_rolling_mean_7d missing train : 0.0089
tma_rolling_mean_7d missing test  : 1.0000


In [23]:
FEATURES_S2 = [
    # Temporal
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    # Identity
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    # TMA anchor features (NEW)
    "tma_last_known",
    "tma_lag1",
    "tma_rolling_mean_7d",
    # Exogenous
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    # Engineered
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    # Horizon
    "horizon_days", "horizon_bucket",
]
CAT_FEATURES = ["nama_pos", "horizon_bucket"]

LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        127,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      1,
    "lambda_l1":         0.1,
    "lambda_l2":         0.1,
    "verbose":           -1,
    "seed":              42,
    "deterministic":     True,
}

FOLDS = [
    ("2024-07-31 18:00:00", "2024-08-01 06:00:00", "2024-10-31 18:00:00"),
    ("2024-10-31 18:00:00", "2024-11-01 06:00:00", "2025-01-31 18:00:00"),
    ("2025-01-31 18:00:00", "2025-03-01 06:00:00", "2025-06-30 18:00:00"),
    ("2025-06-30 18:00:00", "2025-07-01 06:00:00", "2025-09-18 18:00:00"),
]

GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")
ROLLING_COLS = {
    "rolling_rain_48h":     48,
    "rolling_rain_72h":     72,
    "rolling_rain_7d":      168,
    "tma_lag1":             8,
    "tma_rolling_mean_7d":  168,
}

oof_preds_s2 = np.zeros(len(train_fe))
oof_mask_s2  = np.zeros(len(train_fe), dtype=bool)
best_iterations_s2 = []

train_fe = train_fe.sort_values("datetime").reset_index(drop=True)

for fold_idx, (train_cut, val_start, val_end) in enumerate(FOLDS):
    train_cut = pd.Timestamp(train_cut)
    val_start = pd.Timestamp(val_start)
    val_end   = pd.Timestamp(val_end)

    tr_mask  = train_fe["datetime"] <= train_cut
    val_mask = (train_fe["datetime"] >= val_start) & (train_fe["datetime"] <= val_end)

    X_tr  = train_fe.loc[tr_mask,  FEATURES_S2].copy()
    y_tr  = train_fe.loc[tr_mask,  "tma_mdpl"]
    X_val = train_fe.loc[val_mask, FEATURES_S2].copy()
    y_val = train_fe.loc[val_mask, "tma_mdpl"]

    # Per-fold pos stats
    fold_train_df = train_fe.loc[tr_mask].copy()
    pos_stats_fold = fold_train_df.groupby("nama_pos")["tma_mdpl"].agg(
        tma_mean_pos="mean",
        tma_std_pos="std"
    )
    ac_fold = fold_train_df.groupby("nama_pos").apply(
        lambda g: g.sort_values("datetime")["tma_mdpl"].dropna().autocorr(lag=1)
    ).rename("ac_lag1_pos")
    pos_stats_fold = pos_stats_fold.join(ac_fold)

    for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
        X_tr[col]  = train_fe.loc[tr_mask,  "nama_pos"].map(pos_stats_fold[col]).values
        X_val[col] = train_fe.loc[val_mask, "nama_pos"].map(pos_stats_fold[col]).values

    # tma_last_known per fold — TMA terakhir di train fold ini
    last_tma_fold = (
        fold_train_df.sort_values("datetime")
        .groupby("nama_pos")["tma_mdpl"]
        .last()
    )
    X_tr["tma_last_known"]  = train_fe.loc[tr_mask,  "nama_pos"].map(last_tma_fold).values
    X_val["tma_last_known"] = train_fe.loc[val_mask, "nama_pos"].map(last_tma_fold).values

    # Gap masking rolling cols di val
    for col, w in ROLLING_COLS.items():
        if col not in FEATURES_S2:
            continue
        corrupt = (
            (train_fe.loc[val_mask, "datetime"] >= GAP_START) &
            (train_fe.loc[val_mask, "datetime"] <= GAP_END + pd.Timedelta(hours=w))
        )
        X_val.loc[corrupt[corrupt].index, col] = np.nan

    for col in CAT_FEATURES:
        X_tr[col]  = X_tr[col].astype("category")
        X_val[col] = X_val[col].astype("category")

    ds_tr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=CAT_FEATURES, free_raw_data=False)
    ds_val = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_FEATURES, free_raw_data=False)

    model = lgb.train(
        LGB_PARAMS,
        ds_tr,
        num_boost_round=2000,
        valid_sets=[ds_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iterations_s2.append(model.best_iteration)
    preds = model.predict(X_val)
    oof_preds_s2[val_mask] = preds
    oof_mask_s2[val_mask]  = True

    # PERBAIKAN: gunakan np.sqrt(mean_squared_error) sebagai pengganti squared=False
    fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
    print(f"Fold {fold_idx+1} RMSE: {fold_rmse:.4f} | best_iter: {model.best_iteration}")

# OOF RMSE
oof_true_s2 = train_fe.loc[oof_mask_s2, "tma_mdpl"].values
oof_pred_s2 = oof_preds_s2[oof_mask_s2]
# PERBAIKAN:
oof_rmse_s2 = np.sqrt(mean_squared_error(oof_true_s2, oof_pred_s2))
print(f"\n=== OOF RMSE (Slot 2): {oof_rmse_s2:.4f} ===")

# Per-pos RMSE
pos_col_s2 = train_fe.loc[oof_mask_s2, "nama_pos"].values
rmse_per_pos_s2 = {}
for pos in np.unique(pos_col_s2):
    mask = pos_col_s2 == pos
    # PERBAIKAN:
    rmse_per_pos_s2[pos] = np.sqrt(mean_squared_error(oof_true_s2[mask], oof_pred_s2[mask]))
rmse_pos_df_s2 = pd.DataFrame.from_dict(
    rmse_per_pos_s2, orient="index", columns=["rmse"]
).sort_values("rmse", ascending=False)
print("\n=== RMSE per Pos — Slot 2 (top 10 worst) ===")
print(rmse_pos_df_s2.head(10).round(4).to_string())
print("\n=== Slot 1 vs Slot 2 — semua pos ===")
compare_df = rmse_pos_df.rename(columns={"rmse": "slot1"}).join(
    rmse_pos_df_s2.rename(columns={"rmse": "slot2"})
)
compare_df["delta"] = compare_df["slot2"] - compare_df["slot1"]
print(compare_df.sort_values("delta").round(4).to_string())

# Retrain full
mean_best_iter_s2 = int(np.mean(best_iterations_s2) * 1.1)
print(f"\nRetraining full data, num_boost_round={mean_best_iter_s2}...")

X_full_s2 = train_fe[FEATURES_S2].copy()
y_full     = train_fe["tma_mdpl"]
for col in CAT_FEATURES:
    X_full_s2[col] = X_full_s2[col].astype("category")

ds_full_s2 = lgb.Dataset(X_full_s2, label=y_full, categorical_feature=CAT_FEATURES)
final_model_s2 = lgb.train(LGB_PARAMS, ds_full_s2, num_boost_round=mean_best_iter_s2)
final_model_s2.save_model(f"model_s9_exp002_cv{oof_rmse_s2:.4f}.txt")
np.save("oof_exp002.npy", oof_preds_s2)

# Inference
X_test_s2 = test_fe[FEATURES_S2].copy()
for col in CAT_FEATURES:
    X_test_s2[col] = X_test_s2[col].astype("category")

test_preds_s2 = final_model_s2.predict(X_test_s2)

sub_s2 = pd.read_csv("sample_submission.csv")
sub_s2["tma_mdpl"] = test_preds_s2

print(f"\n=== SUBMISSION SANITY CHECK Slot 2 ===")
print(f"Rows      : {len(sub_s2)}")
print(f"NaN       : {sub_s2['tma_mdpl'].isna().sum()}")
print(f"Inf       : {np.isinf(sub_s2['tma_mdpl']).sum()}")
print(f"Min pred  : {sub_s2['tma_mdpl'].min():.4f}")
print(f"Max pred  : {sub_s2['tma_mdpl'].max():.4f}")
print(f"Mean pred : {sub_s2['tma_mdpl'].mean():.4f}")

sub_s2.to_csv(f"sub_exp002_slot2.csv", index=False)
print(f"Saved: sub_exp002_slot2.csv")

Fold 1 RMSE: 0.6091 | best_iter: 108
Fold 2 RMSE: 0.7181 | best_iter: 111
Fold 3 RMSE: 1.1673 | best_iter: 119
Fold 4 RMSE: 0.8511 | best_iter: 106

=== OOF RMSE (Slot 2): 0.8900 ===

=== RMSE per Pos — Slot 2 (top 10 worst) ===
                            rmse
Peren                     2.5539
Karangnongko              1.8525
Floodway Bridge C         1.4803
Ketonggo                  1.2771
Bojonegoro - Kali Kethek  1.2252
Napel                     1.1764
Kajangan                  0.9580
Kedungupit                0.7936
Wonogiri Dam              0.7369
Jarum                     0.7094

=== Slot 1 vs Slot 2 — semua pos ===
                            slot1   slot2   delta
Gunungsari                 3.1469  0.4170 -2.7299
Wonogiri Dam               2.0993  0.7369 -1.3624
Kali Anyar - Kreteg Abang  1.1175  0.3824 -0.7351
Colo Weir                  1.0905  0.4462 -0.6444
Bengkelolor                1.2065  0.5644 -0.6421
Bojonegoro - Kali Kethek   1.8116  1.2252 -0.5863
Sumberrejo          

In [24]:
# Cek distribusi prediksi Slot 2 per horizon bucket
test_fe["pred_s2"] = test_preds_s2

print("=== Pred distribution per horizon bucket ===")
print(test_fe.groupby("horizon_bucket")["pred_s2"].agg(
    ["mean", "std", "min", "max"]
).round(4))

print("\n=== Pred distribution per pos (worst suspects) ===")
suspects = ["Wonogiri Dam", "Gunungsari", "Colo Weir", "Peren"]
for pos in suspects:
    sub = test_fe[test_fe["nama_pos"] == pos]["pred_s2"]
    print(f"{pos:30s} mean={sub.mean():.2f} std={sub.std():.2f} "
          f"min={sub.min():.2f} max={sub.max():.2f}")

print("\n=== tma_last_known per pos (anchor values) ===")
anchor = test_fe.groupby("nama_pos")["tma_last_known"].first().sort_values(ascending=False)
print(anchor.round(3).to_string())

=== Pred distribution per horizon bucket ===
                  mean      std     min       max
horizon_bucket                                   
far             55.922  46.3912  3.5452  142.3735
mid             55.878  46.4012  3.7205  142.1727
near            55.542  46.4790  3.5661  140.7994

=== Pred distribution per pos (worst suspects) ===
Wonogiri Dam                   mean=133.31 std=0.44 min=132.40 max=135.16
Gunungsari                     mean=11.01 std=0.66 min=9.99 max=13.96
Colo Weir                      mean=107.90 std=0.23 min=107.34 max=109.45
Peren                          mean=91.26 std=0.61 min=90.51 max=93.84

=== tma_last_known per pos (anchor values) ===
nama_pos
Ngadipiro                    143.440
Ngrembang                    139.910
Wonogiri Dam                 129.870
Badegan                      121.990
Colo Weir                    107.970
Kali Pepe - Tugu Boto         94.640
Peren                         91.400
Jarum                         90.240
Sekayu     

In [25]:
# Cek apakah urutan id di submission match dengan test_fe
sub_check = pd.read_csv("sample_submission.csv")

# Reconstruct id dari test_fe untuk cross-check
test_fe["id_reconstructed"] = (
    test_fe["datetime"].astype(str) + " - " + test_fe["nama_pos"]
)

# Cek berapa id yang match
match = (sub_check["id"].values == test_fe["id_reconstructed"].values)
print(f"Total rows     : {len(sub_check)}")
print(f"ID match       : {match.sum()}")
print(f"ID mismatch    : {(~match).sum()}")

# Sample mismatch kalau ada
if (~match).sum() > 0:
    mismatch_idx = np.where(~match)[0][:5]
    print("\nContoh mismatch:")
    for i in mismatch_idx:
        print(f"  sub id   : {sub_check['id'].iloc[i]}")
        print(f"  test_fe  : {test_fe['id_reconstructed'].iloc[i]}")

Total rows     : 21780
ID match       : 21780
ID mismatch    : 0


In [26]:
# ── EXP003 — Feature tambahan ────────────────────────────────────────────────

# 1. Rolling rain 14d dan 30d
train_fe = train_fe.sort_values(["nama_pos", "datetime"])
test_fe  = test_fe.sort_values(["nama_pos", "datetime"])

for df in [train_fe, test_fe]:
    df["rolling_rain_14d"] = df.groupby("nama_pos")["rainfall_mm"].transform(
        lambda x: x.rolling(336, min_periods=1).sum()
    )
    df["rolling_rain_30d"] = df.groupby("nama_pos")["rainfall_mm"].transform(
        lambda x: x.rolling(720, min_periods=1).sum()
    )

# Gap masking rolling baru
GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")

for col, w in [("rolling_rain_14d", 336), ("rolling_rain_30d", 720)]:
    corrupt = (
        (train_fe["datetime"] >= GAP_START) &
        (train_fe["datetime"] <= GAP_END + pd.Timedelta(hours=w))
    )
    train_fe.loc[corrupt, col] = np.nan
    # test tidak kena gap — tidak perlu mask

# 2. Interaction features
for df in [train_fe, test_fe]:
    # Soil moisture jenuh × hujan intensif = kondisi banjir
    df["sm_x_rain24h"] = df["soil_moisture_7_28cm"] * df["rainfall_max_24h_mm"]
    df["sm_x_rain7d"]  = df["soil_moisture_7_28cm"] * df["rolling_rain_7d"]
    # Musiman per karakter pos
    df["bulan_x_ac"]   = df["bulan"] * df["ac_lag1_pos"]

print("=== EXP003 feature check ===")
for col in ["rolling_rain_14d", "rolling_rain_30d", "sm_x_rain24h", "sm_x_rain7d", "bulan_x_ac"]:
    print(f"{col:25s} train_missing={train_fe[col].isna().mean():.4f} "
          f"test_missing={test_fe[col].isna().mean():.4f}")

=== EXP003 feature check ===
rolling_rain_14d          train_missing=0.0156 test_missing=0.0000
rolling_rain_30d          train_missing=0.0327 test_missing=0.0000
sm_x_rain24h              train_missing=0.0000 test_missing=0.0000
sm_x_rain7d               train_missing=0.0081 test_missing=0.0000
bulan_x_ac                train_missing=0.0000 test_missing=0.0000


In [29]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as _mse

def rmse(y_true, y_pred):
    return np.sqrt(_mse(y_true, y_pred))

FEATURES_S3 = [
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    "rolling_rain_14d", "rolling_rain_30d",
    "sm_x_rain24h", "sm_x_rain7d", "bulan_x_ac",
    "horizon_days", "horizon_bucket",
]
CAT_FEATURES = ["nama_pos", "horizon_bucket"]

LGB_PARAMS_S3 = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.03,
    "num_leaves":        255,
    "min_child_samples": 15,
    "feature_fraction":  0.7,
    "bagging_fraction":  0.7,
    "bagging_freq":      1,
    "lambda_l1":         0.05,
    "lambda_l2":         0.05,
    "verbose":           -1,
    "seed":              42,
    "deterministic":     True,
}

FOLDS = [
    ("2024-07-31 18:00:00", "2024-08-01 06:00:00", "2024-10-31 18:00:00"),
    ("2024-10-31 18:00:00", "2024-11-01 06:00:00", "2025-01-31 18:00:00"),
    ("2025-01-31 18:00:00", "2025-03-01 06:00:00", "2025-06-30 18:00:00"),
    ("2025-06-30 18:00:00", "2025-07-01 06:00:00", "2025-09-18 18:00:00"),
]

GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")
ROLLING_COLS_S3 = {
    "rolling_rain_48h":  48,
    "rolling_rain_72h":  72,
    "rolling_rain_7d":   168,
    "rolling_rain_14d":  336,
    "rolling_rain_30d":  720,
}

oof_preds_s3 = np.zeros(len(train_fe))
oof_mask_s3  = np.zeros(len(train_fe), dtype=bool)
best_iterations_s3 = []

train_fe = train_fe.sort_values("datetime").reset_index(drop=True)

for fold_idx, (train_cut, val_start, val_end) in enumerate(FOLDS):
    train_cut = pd.Timestamp(train_cut)
    val_start = pd.Timestamp(val_start)
    val_end   = pd.Timestamp(val_end)

    tr_mask  = train_fe["datetime"] <= train_cut
    val_mask = (train_fe["datetime"] >= val_start) & (train_fe["datetime"] <= val_end)

    X_tr  = train_fe.loc[tr_mask,  FEATURES_S3].copy()
    y_tr  = train_fe.loc[tr_mask,  "tma_mdpl"]
    X_val = train_fe.loc[val_mask, FEATURES_S3].copy()
    y_val = train_fe.loc[val_mask, "tma_mdpl"]

    # Per-fold pos stats
    fold_train_df = train_fe.loc[tr_mask].copy()
    pos_stats_fold = fold_train_df.groupby("nama_pos")["tma_mdpl"].agg(
        tma_mean_pos="mean",
        tma_std_pos="std"
    )
    ac_fold = fold_train_df.groupby("nama_pos").apply(
        lambda g: g.sort_values("datetime")["tma_mdpl"].dropna().autocorr(lag=1)
    ).rename("ac_lag1_pos")
    pos_stats_fold = pos_stats_fold.join(ac_fold)

    for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
        X_tr[col]  = train_fe.loc[tr_mask,  "nama_pos"].map(pos_stats_fold[col]).values
        X_val[col] = train_fe.loc[val_mask, "nama_pos"].map(pos_stats_fold[col]).values

    # Recompute bulan_x_ac dengan ac per fold
    X_tr["bulan_x_ac"]  = X_tr["bulan"] * X_tr["ac_lag1_pos"]
    X_val["bulan_x_ac"] = X_val["bulan"] * X_val["ac_lag1_pos"]

    # Gap masking
    for col, w in ROLLING_COLS_S3.items():
        if col not in FEATURES_S3:
            continue
        corrupt = (
            (train_fe.loc[val_mask, "datetime"] >= GAP_START) &
            (train_fe.loc[val_mask, "datetime"] <= GAP_END + pd.Timedelta(hours=w))
        )
        X_val.loc[corrupt[corrupt].index, col] = np.nan

    for col in CAT_FEATURES:
        X_tr[col]  = X_tr[col].astype("category")
        X_val[col] = X_val[col].astype("category")

    ds_tr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=CAT_FEATURES, free_raw_data=False)
    ds_val = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_FEATURES, free_raw_data=False)

    model = lgb.train(
        LGB_PARAMS_S3,
        ds_tr,
        num_boost_round=3000,
        valid_sets=[ds_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=75, verbose=False),
            lgb.log_evaluation(period=300),
        ],
    )

    best_iterations_s3.append(model.best_iteration)
    preds = model.predict(X_val)
    oof_preds_s3[val_mask] = preds
    oof_mask_s3[val_mask]  = True

    fold_rmse = rmse(y_val, preds)
    print(f"Fold {fold_idx+1} RMSE: {fold_rmse:.4f} | best_iter: {model.best_iteration}")

# OOF RMSE
oof_true_s3 = train_fe.loc[oof_mask_s3, "tma_mdpl"].values
oof_pred_s3 = oof_preds_s3[oof_mask_s3]
oof_rmse_s3 = rmse(oof_true_s3, oof_pred_s3)
print(f"\n=== OOF RMSE (exp003): {oof_rmse_s3:.4f} ===")
print(f"    vs Slot 1        : 1.2888")
print(f"    Delta            : {oof_rmse_s3 - 1.2888:+.4f}")

# Per-pos RMSE
pos_col_s3 = train_fe.loc[oof_mask_s3, "nama_pos"].values
rmse_per_pos_s3 = {}
for pos in np.unique(pos_col_s3):
    mask = pos_col_s3 == pos
    rmse_per_pos_s3[pos] = rmse(oof_true_s3[mask], oof_pred_s3[mask])

rmse_pos_df_s3 = pd.DataFrame.from_dict(
    rmse_per_pos_s3, orient="index", columns=["rmse"]
).sort_values("rmse", ascending=False)

print("\n=== RMSE per Pos — exp003 vs Slot 1 ===")
compare_s3 = rmse_pos_df.rename(columns={"rmse": "slot1"}).join(
    rmse_pos_df_s3.rename(columns={"rmse": "exp003"})
)
compare_s3["delta"] = compare_s3["exp003"] - compare_s3["slot1"]
print(compare_s3.sort_values("delta").round(4).to_string())

# Retrain full
mean_best_iter_s3 = int(np.mean(best_iterations_s3) * 1.1)
print(f"\nRetraining full data, num_boost_round={mean_best_iter_s3}...")

X_full_s3 = train_fe[FEATURES_S3].copy()
y_full     = train_fe["tma_mdpl"]
X_full_s3["bulan_x_ac"] = X_full_s3["bulan"] * X_full_s3["ac_lag1_pos"]
for col in CAT_FEATURES:
    X_full_s3[col] = X_full_s3[col].astype("category")

ds_full_s3 = lgb.Dataset(X_full_s3, label=y_full, categorical_feature=CAT_FEATURES)
final_model_s3 = lgb.train(LGB_PARAMS_S3, ds_full_s3, num_boost_round=mean_best_iter_s3)
final_model_s3.save_model(f"model_s9_exp003_cv{oof_rmse_s3:.4f}.txt")
np.save("oof_exp003.npy", oof_preds_s3)

# Inference
X_test_s3 = test_fe[FEATURES_S3].copy()
X_test_s3["bulan_x_ac"] = X_test_s3["bulan"] * X_test_s3["ac_lag1_pos"]
for col in CAT_FEATURES:
    X_test_s3[col] = X_test_s3[col].astype("category")

test_preds_s3 = final_model_s3.predict(X_test_s3)

sub_s3 = pd.read_csv("sample_submission.csv")
sub_s3["tma_mdpl"] = test_preds_s3

print(f"\n=== SUBMISSION SANITY CHECK exp003 ===")
print(f"Rows      : {len(sub_s3)}")
print(f"NaN       : {sub_s3['tma_mdpl'].isna().sum()}")
print(f"Inf       : {np.isinf(sub_s3['tma_mdpl']).sum()}")
print(f"Min pred  : {sub_s3['tma_mdpl'].min():.4f}")
print(f"Max pred  : {sub_s3['tma_mdpl'].max():.4f}")
print(f"Mean pred : {sub_s3['tma_mdpl'].mean():.4f}")

sub_s3.to_csv("sub_exp003.csv", index=False)
print("Saved: sub_exp003.csv")

Fold 1 RMSE: 1.4988 | best_iter: 149
Fold 2 RMSE: 1.4017 | best_iter: 167
Fold 3 RMSE: 1.4100 | best_iter: 187
Fold 4 RMSE: 0.9406 | best_iter: 186

=== OOF RMSE (exp003): 1.3473 ===
    vs Slot 1        : 1.2888
    Delta            : +0.0585

=== RMSE per Pos — exp003 vs Slot 1 ===
                            slot1  exp003   delta
Gunungsari                 3.1469  3.0588 -0.0881
Kajangan                   1.0533  0.9743 -0.0789
Sumberrejo                 1.0772  1.0370 -0.0402
Kali Pepe - Tugu Boto      0.5864  0.5477 -0.0387
Jarum                      0.8278  0.7893 -0.0384
Bengkelolor                1.2065  1.1753 -0.0312
Sekayu                     0.7087  0.6805 -0.0283
Boboh Kali Lamong          1.0395  1.0192 -0.0204
Lorog                      0.4171  0.3975 -0.0196
Jurug                      0.8504  0.8334 -0.0170
Cepu                       0.9671  0.9684  0.0013
Kali Pepe - PTPN           0.4135  0.4247  0.0112
Ketonggo                   1.3347  1.3550  0.0204
Floodway Bridge

In [30]:
# ── EXP004 — Slot 1 features + sample weight untuk pos spike ─────────────────

FEATURES_S1 = [
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    "horizon_days", "horizon_bucket",
]
CAT_FEATURES = ["nama_pos", "horizon_bucket"]

SPIKE_POS = [
    "Peren", "Karangnongko", "Napel",
    "Bojonegoro - Kali Kethek", "Jarum", "Kali Anyar - Kreteg Abang"
]

LGB_PARAMS_S4 = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        127,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      1,
    "lambda_l1":         0.1,
    "lambda_l2":         0.1,
    "verbose":           -1,
    "seed":              42,
    "deterministic":     True,
}

FOLDS = [
    ("2024-07-31 18:00:00", "2024-08-01 06:00:00", "2024-10-31 18:00:00"),
    ("2024-10-31 18:00:00", "2024-11-01 06:00:00", "2025-01-31 18:00:00"),
    ("2025-01-31 18:00:00", "2025-03-01 06:00:00", "2025-06-30 18:00:00"),
    ("2025-06-30 18:00:00", "2025-07-01 06:00:00", "2025-09-18 18:00:00"),
]

GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")
ROLLING_COLS = {
    "rolling_rain_48h": 48,
    "rolling_rain_72h": 72,
    "rolling_rain_7d":  168,
}

oof_preds_s4 = np.zeros(len(train_fe))
oof_mask_s4  = np.zeros(len(train_fe), dtype=bool)
best_iterations_s4 = []

train_fe = train_fe.sort_values("datetime").reset_index(drop=True)

for fold_idx, (train_cut, val_start, val_end) in enumerate(FOLDS):
    train_cut = pd.Timestamp(train_cut)
    val_start = pd.Timestamp(val_start)
    val_end   = pd.Timestamp(val_end)

    tr_mask  = train_fe["datetime"] <= train_cut
    val_mask = (train_fe["datetime"] >= val_start) & (train_fe["datetime"] <= val_end)

    X_tr  = train_fe.loc[tr_mask,  FEATURES_S1].copy()
    y_tr  = train_fe.loc[tr_mask,  "tma_mdpl"]
    X_val = train_fe.loc[val_mask, FEATURES_S1].copy()
    y_val = train_fe.loc[val_mask, "tma_mdpl"]

    # Sample weight — spike pos dapat 3x
    w_tr = train_fe.loc[tr_mask, "nama_pos"].isin(SPIKE_POS).astype(float)
    w_tr = w_tr.replace({0.0: 1.0, 1.0: 3.0}).values

    # Per-fold pos stats
    fold_train_df = train_fe.loc[tr_mask].copy()
    pos_stats_fold = fold_train_df.groupby("nama_pos")["tma_mdpl"].agg(
        tma_mean_pos="mean",
        tma_std_pos="std"
    )
    ac_fold = fold_train_df.groupby("nama_pos").apply(
        lambda g: g.sort_values("datetime")["tma_mdpl"].dropna().autocorr(lag=1)
    ).rename("ac_lag1_pos")
    pos_stats_fold = pos_stats_fold.join(ac_fold)

    for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
        X_tr[col]  = train_fe.loc[tr_mask,  "nama_pos"].map(pos_stats_fold[col]).values
        X_val[col] = train_fe.loc[val_mask, "nama_pos"].map(pos_stats_fold[col]).values

    # Gap masking
    for col, w in ROLLING_COLS.items():
        corrupt = (
            (train_fe.loc[val_mask, "datetime"] >= GAP_START) &
            (train_fe.loc[val_mask, "datetime"] <= GAP_END + pd.Timedelta(hours=w))
        )
        X_val.loc[corrupt[corrupt].index, col] = np.nan

    for col in CAT_FEATURES:
        X_tr[col]  = X_tr[col].astype("category")
        X_val[col] = X_val[col].astype("category")

    ds_tr  = lgb.Dataset(
        X_tr, label=y_tr,
        weight=w_tr,
        categorical_feature=CAT_FEATURES,
        free_raw_data=False
    )
    ds_val = lgb.Dataset(
        X_val, label=y_val,
        categorical_feature=CAT_FEATURES,
        free_raw_data=False
    )

    model = lgb.train(
        LGB_PARAMS_S4,
        ds_tr,
        num_boost_round=2000,
        valid_sets=[ds_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iterations_s4.append(model.best_iteration)
    preds = model.predict(X_val)
    oof_preds_s4[val_mask] = preds
    oof_mask_s4[val_mask]  = True

    fold_rmse = rmse(y_val, preds)
    print(f"Fold {fold_idx+1} RMSE: {fold_rmse:.4f} | best_iter: {model.best_iteration}")

# OOF RMSE
oof_true_s4 = train_fe.loc[oof_mask_s4, "tma_mdpl"].values
oof_pred_s4 = oof_preds_s4[oof_mask_s4]
oof_rmse_s4 = rmse(oof_true_s4, oof_pred_s4)
print(f"\n=== OOF RMSE (exp004): {oof_rmse_s4:.4f} ===")
print(f"    vs Slot 1        : 1.2888")
print(f"    Delta            : {oof_rmse_s4 - 1.2888:+.4f}")

# Per-pos RMSE
pos_col_s4 = train_fe.loc[oof_mask_s4, "nama_pos"].values
rmse_per_pos_s4 = {}
for pos in np.unique(pos_col_s4):
    mask = pos_col_s4 == pos
    rmse_per_pos_s4[pos] = rmse(oof_true_s4[mask], oof_pred_s4[mask])

rmse_pos_df_s4 = pd.DataFrame.from_dict(
    rmse_per_pos_s4, orient="index", columns=["rmse"]
).sort_values("rmse", ascending=False)

print("\n=== RMSE per Pos — exp004 vs Slot 1 ===")
compare_s4 = rmse_pos_df.rename(columns={"rmse": "slot1"}).join(
    rmse_pos_df_s4.rename(columns={"rmse": "exp004"})
)
compare_s4["delta"] = compare_s4["exp004"] - compare_s4["slot1"]
print(compare_s4.sort_values("delta").round(4).to_string())

# Cek spike pos khusus
print("\n=== Spike pos detail ===")
for pos in SPIKE_POS:
    s1 = compare_s4.loc[pos, "slot1"] if pos in compare_s4.index else "N/A"
    s4 = compare_s4.loc[pos, "exp004"] if pos in compare_s4.index else "N/A"
    d  = compare_s4.loc[pos, "delta"] if pos in compare_s4.index else "N/A"
    print(f"  {pos:35s} slot1={s1:.4f} exp004={s4:.4f} delta={d:+.4f}")

# Retrain full
mean_best_iter_s4 = int(np.mean(best_iterations_s4) * 1.1)
print(f"\nRetraining full data, num_boost_round={mean_best_iter_s4}...")

X_full_s4 = train_fe[FEATURES_S1].copy()
y_full     = train_fe["tma_mdpl"]
w_full     = train_fe["nama_pos"].isin(SPIKE_POS).astype(float).replace({0.0: 1.0, 1.0: 3.0}).values

for col in CAT_FEATURES:
    X_full_s4[col] = X_full_s4[col].astype("category")

ds_full_s4 = lgb.Dataset(
    X_full_s4, label=y_full,
    weight=w_full,
    categorical_feature=CAT_FEATURES
)
final_model_s4 = lgb.train(LGB_PARAMS_S4, ds_full_s4, num_boost_round=mean_best_iter_s4)
final_model_s4.save_model(f"model_s9_exp004_cv{oof_rmse_s4:.4f}.txt")
np.save("oof_exp004.npy", oof_preds_s4)

# Inference
X_test_s4 = test_fe[FEATURES_S1].copy()
for col in CAT_FEATURES:
    X_test_s4[col] = X_test_s4[col].astype("category")

test_preds_s4 = final_model_s4.predict(X_test_s4)

sub_s4 = pd.read_csv("sample_submission.csv")
sub_s4["tma_mdpl"] = test_preds_s4

print(f"\n=== SUBMISSION SANITY CHECK exp004 ===")
print(f"Rows      : {len(sub_s4)}")
print(f"NaN       : {sub_s4['tma_mdpl'].isna().sum()}")
print(f"Inf       : {np.isinf(sub_s4['tma_mdpl']).sum()}")
print(f"Min pred  : {sub_s4['tma_mdpl'].min():.4f}")
print(f"Max pred  : {sub_s4['tma_mdpl'].max():.4f}")
print(f"Mean pred : {sub_s4['tma_mdpl'].mean():.4f}")

sub_s4.to_csv("sub_exp004.csv", index=False)
print("Saved: sub_exp004.csv")

Fold 1 RMSE: 1.7259 | best_iter: 87
Fold 2 RMSE: 1.4718 | best_iter: 112
Fold 3 RMSE: 1.4131 | best_iter: 96
Fold 4 RMSE: 1.0525 | best_iter: 103

=== OOF RMSE (exp004): 1.4445 ===
    vs Slot 1        : 1.2888
    Delta            : +0.1557

=== RMSE per Pos — exp004 vs Slot 1 ===
                            slot1  exp004   delta
Floodway Bridge C          1.8536  1.8429 -0.0107
Kedungupit                 1.1496  1.1431 -0.0066
Kajangan                   1.0533  1.0624  0.0091
Boboh Kali Lamong          1.0395  1.0489  0.0093
Jurug                      0.8504  0.8682  0.0178
Kali Pepe - PTPN           0.4135  0.4316  0.0181
Jarum                      0.8278  0.8509  0.0231
Sumberrejo                 1.0772  1.1209  0.0437
Colo Weir                  1.0905  1.1379  0.0474
Karangnongko               1.7051  1.7640  0.0589
Karanggeneng               0.7955  0.8605  0.0650
Babat                      0.8282  0.8964  0.0682
Arjowinangun - Pacitan     0.4927  0.5719  0.0792
Cepu             

In [31]:
# ── EXP005 — Slot 1 + Target Encoding (nama_pos × bulan) ─────────────────────

FEATURES_S5 = [
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    # Target encoding baru
    "tma_pos_bulan_mean",
    "tma_pos_bulan_std",
    "tma_pos_hour_mean",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    "horizon_days", "horizon_bucket",
]
CAT_FEATURES = ["nama_pos", "horizon_bucket"]

LGB_PARAMS_S5 = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        127,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      1,
    "lambda_l1":         0.1,
    "lambda_l2":         0.1,
    "verbose":           -1,
    "seed":              42,
    "deterministic":     True,
}

FOLDS = [
    ("2024-07-31 18:00:00", "2024-08-01 06:00:00", "2024-10-31 18:00:00"),
    ("2024-10-31 18:00:00", "2024-11-01 06:00:00", "2025-01-31 18:00:00"),
    ("2025-01-31 18:00:00", "2025-03-01 06:00:00", "2025-06-30 18:00:00"),
    ("2025-06-30 18:00:00", "2025-07-01 06:00:00", "2025-09-18 18:00:00"),
]

GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")
ROLLING_COLS = {
    "rolling_rain_48h": 48,
    "rolling_rain_72h": 72,
    "rolling_rain_7d":  168,
}

oof_preds_s5 = np.zeros(len(train_fe))
oof_mask_s5  = np.zeros(len(train_fe), dtype=bool)
best_iterations_s5 = []

train_fe = train_fe.sort_values("datetime").reset_index(drop=True)

for fold_idx, (train_cut, val_start, val_end) in enumerate(FOLDS):
    train_cut = pd.Timestamp(train_cut)
    val_start = pd.Timestamp(val_start)
    val_end   = pd.Timestamp(val_end)

    tr_mask  = train_fe["datetime"] <= train_cut
    val_mask = (train_fe["datetime"] >= val_start) & (train_fe["datetime"] <= val_end)

    X_tr  = train_fe.loc[tr_mask,  FEATURES_S5].copy()
    y_tr  = train_fe.loc[tr_mask,  "tma_mdpl"]
    X_val = train_fe.loc[val_mask, FEATURES_S5].copy()
    y_val = train_fe.loc[val_mask, "tma_mdpl"]

    fold_train_df = train_fe.loc[tr_mask].copy()

    # Per-fold pos stats
    pos_stats_fold = fold_train_df.groupby("nama_pos")["tma_mdpl"].agg(
        tma_mean_pos="mean",
        tma_std_pos="std"
    )
    ac_fold = fold_train_df.groupby("nama_pos").apply(
        lambda g: g.sort_values("datetime")["tma_mdpl"].dropna().autocorr(lag=1)
    ).rename("ac_lag1_pos")
    pos_stats_fold = pos_stats_fold.join(ac_fold)

    for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
        X_tr[col]  = train_fe.loc[tr_mask,  "nama_pos"].map(pos_stats_fold[col]).values
        X_val[col] = train_fe.loc[val_mask, "nama_pos"].map(pos_stats_fold[col]).values

    # Target encoding per fold — LEAKAGE SAFE
    # 1. pos × bulan mean & std
    te_pos_bulan = fold_train_df.groupby(
        ["nama_pos", "bulan"]
    )["tma_mdpl"].agg(
        tma_pos_bulan_mean="mean",
        tma_pos_bulan_std="std"
    ).reset_index()

    # 2. pos × hour mean
    te_pos_hour = fold_train_df.groupby(
        ["nama_pos", "hour"]
    )["tma_mdpl"].agg(
        tma_pos_hour_mean="mean"
    ).reset_index()

    # Merge ke X_tr
    tr_df = train_fe.loc[tr_mask, ["nama_pos", "bulan", "hour"]].copy()
    tr_df = tr_df.merge(te_pos_bulan, on=["nama_pos", "bulan"], how="left")
    tr_df = tr_df.merge(te_pos_hour,  on=["nama_pos", "hour"],  how="left")
    X_tr["tma_pos_bulan_mean"] = tr_df["tma_pos_bulan_mean"].values
    X_tr["tma_pos_bulan_std"]  = tr_df["tma_pos_bulan_std"].values
    X_tr["tma_pos_hour_mean"]  = tr_df["tma_pos_hour_mean"].values

    # Merge ke X_val
    val_df = train_fe.loc[val_mask, ["nama_pos", "bulan", "hour"]].copy()
    val_df = val_df.merge(te_pos_bulan, on=["nama_pos", "bulan"], how="left")
    val_df = val_df.merge(te_pos_hour,  on=["nama_pos", "hour"],  how="left")
    X_val["tma_pos_bulan_mean"] = val_df["tma_pos_bulan_mean"].values
    X_val["tma_pos_bulan_std"]  = val_df["tma_pos_bulan_std"].values
    X_val["tma_pos_hour_mean"]  = val_df["tma_pos_hour_mean"].values

    # Gap masking
    for col, w in ROLLING_COLS.items():
        corrupt = (
            (train_fe.loc[val_mask, "datetime"] >= GAP_START) &
            (train_fe.loc[val_mask, "datetime"] <= GAP_END + pd.Timedelta(hours=w))
        )
        X_val.loc[corrupt[corrupt].index, col] = np.nan

    for col in CAT_FEATURES:
        X_tr[col]  = X_tr[col].astype("category")
        X_val[col] = X_val[col].astype("category")

    ds_tr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=CAT_FEATURES, free_raw_data=False)
    ds_val = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_FEATURES, free_raw_data=False)

    model = lgb.train(
        LGB_PARAMS_S5,
        ds_tr,
        num_boost_round=2000,
        valid_sets=[ds_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )

    best_iterations_s5.append(model.best_iteration)
    preds = model.predict(X_val)
    oof_preds_s5[val_mask] = preds
    oof_mask_s5[val_mask]  = True

    fold_rmse = rmse(y_val, preds)
    print(f"Fold {fold_idx+1} RMSE: {fold_rmse:.4f} | best_iter: {model.best_iteration}")

# OOF RMSE
oof_true_s5 = train_fe.loc[oof_mask_s5, "tma_mdpl"].values
oof_pred_s5 = oof_preds_s5[oof_mask_s5]
oof_rmse_s5 = rmse(oof_true_s5, oof_pred_s5)
print(f"\n=== OOF RMSE (exp005): {oof_rmse_s5:.4f} ===")
print(f"    vs Slot 1        : 1.2888")
print(f"    Delta            : {oof_rmse_s5 - 1.2888:+.4f}")

# Per-pos RMSE
pos_col_s5 = train_fe.loc[oof_mask_s5, "nama_pos"].values
rmse_per_pos_s5 = {}
for pos in np.unique(pos_col_s5):
    mask = pos_col_s5 == pos
    rmse_per_pos_s5[pos] = rmse(oof_true_s5[mask], oof_pred_s5[mask])

rmse_pos_df_s5 = pd.DataFrame.from_dict(
    rmse_per_pos_s5, orient="index", columns=["rmse"]
).sort_values("rmse", ascending=False)

print("\n=== RMSE per Pos — exp005 vs Slot 1 ===")
compare_s5 = rmse_pos_df.rename(columns={"rmse": "slot1"}).join(
    rmse_pos_df_s5.rename(columns={"rmse": "exp005"})
)
compare_s5["delta"] = compare_s5["exp005"] - compare_s5["slot1"]
print(compare_s5.sort_values("delta").round(4).to_string())

# Retrain full — target encoding dari full train
print(f"\nRetraining full data...")
mean_best_iter_s5 = int(np.mean(best_iterations_s5) * 1.1)

# Global target encoding untuk inference
te_pos_bulan_full = train_fe.groupby(
    ["nama_pos", "bulan"]
)["tma_mdpl"].agg(
    tma_pos_bulan_mean="mean",
    tma_pos_bulan_std="std"
).reset_index()

te_pos_hour_full = train_fe.groupby(
    ["nama_pos", "hour"]
)["tma_mdpl"].agg(
    tma_pos_hour_mean="mean"
).reset_index()

def apply_te(df, te_bulan, te_hour):
    df = df.copy()
    tmp = df[["nama_pos", "bulan", "hour"]].merge(
        te_bulan, on=["nama_pos", "bulan"], how="left"
    ).merge(
        te_hour, on=["nama_pos", "hour"], how="left"
    )
    df["tma_pos_bulan_mean"] = tmp["tma_pos_bulan_mean"].values
    df["tma_pos_bulan_std"]  = tmp["tma_pos_bulan_std"].values
    df["tma_pos_hour_mean"]  = tmp["tma_pos_hour_mean"].values
    return df

X_full_s5 = train_fe[FEATURES_S5].copy()
X_full_s5 = apply_te(X_full_s5, te_pos_bulan_full, te_pos_hour_full)
y_full     = train_fe["tma_mdpl"]
for col in CAT_FEATURES:
    X_full_s5[col] = X_full_s5[col].astype("category")

ds_full_s5 = lgb.Dataset(X_full_s5, label=y_full, categorical_feature=CAT_FEATURES)
final_model_s5 = lgb.train(LGB_PARAMS_S5, ds_full_s5, num_boost_round=mean_best_iter_s5)
final_model_s5.save_model(f"model_s9_exp005_cv{oof_rmse_s5:.4f}.txt")
np.save("oof_exp005.npy", oof_preds_s5)

# Inference
X_test_s5 = test_fe[FEATURES_S5].copy()
X_test_s5 = apply_te(X_test_s5, te_pos_bulan_full, te_pos_hour_full)
for col in CAT_FEATURES:
    X_test_s5[col] = X_test_s5[col].astype("category")

test_preds_s5 = final_model_s5.predict(X_test_s5)

sub_s5 = pd.read_csv("sample_submission.csv")
sub_s5["tma_mdpl"] = test_preds_s5

print(f"\n=== SUBMISSION SANITY CHECK exp005 ===")
print(f"Rows      : {len(sub_s5)}")
print(f"NaN       : {sub_s5['tma_mdpl'].isna().sum()}")
print(f"Inf       : {np.isinf(sub_s5['tma_mdpl']).sum()}")
print(f"Min pred  : {sub_s5['tma_mdpl'].min():.4f}")
print(f"Max pred  : {sub_s5['tma_mdpl'].max():.4f}")
print(f"Mean pred : {sub_s5['tma_mdpl'].mean():.4f}")

sub_s5.to_csv("sub_exp005.csv", index=False)
print("Saved: sub_exp005.csv")

KeyError: "['tma_pos_bulan_mean', 'tma_pos_bulan_std', 'tma_pos_hour_mean'] not in index"